In [ ]:
# ============================================================
# KAGGLE-SAFE-01 — GPU + ENVIRONMENT AUDIT
# ============================================================

import os
import sys
import platform
import subprocess

print("=" * 70)
print("KAGGLE-SAFE-01 : ENVIRONMENT AUDIT")
print("=" * 70)

# Python
print("\n[PYTHON]")
print("Version :", sys.version)
print("Platform:", platform.platform())

# PyTorch
print("\n[PYTORCH]")
try:
    import torch

    print("Version      :", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version :", torch.version.cuda)
    print("GPU count    :", torch.cuda.device_count())

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)

            print(f"\nGPU {i}")
            print("Name        :", torch.cuda.get_device_name(i))
            print("VRAM total  :", round(props.total_memory / 1024**3, 2), "GB")
            print("Compute Cap :", f"{props.major}.{props.minor}")

            free, total = torch.cuda.mem_get_info(i)
            print("VRAM free   :", round(free / 1024**3, 2), "GB")
            print("VRAM used   :", round((total-free) / 1024**3, 2), "GB")

except Exception as e:
    print("PyTorch check failed:", repr(e))

# NVIDIA
print("\n[NVIDIA-SMI]")
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        timeout=20
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
except Exception as e:
    print("nvidia-smi failed:", repr(e))

# Important libraries
print("\n[LIBRARIES]")

libraries = [
    "transformers",
    "accelerate",
    "diffusers",
    "safetensors",
    "PIL",
    "numpy",
    "imageio",
    "moviepy",
]

for name in libraries:
    try:
        module = __import__(name)
        version = getattr(module, "__version__", "unknown")
        print(f"{name:15} OK   {version}")
    except Exception as e:
        print(f"{name:15} MISSING   {e}")

# Disk
print("\n[DISK]")
try:
    result = subprocess.run(
        ["df", "-h", "/"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print(result.stdout)
except Exception as e:
    print("Disk check failed:", repr(e))

print("\n" + "=" * 70)
print("KAGGLE-SAFE-01 COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# KAGGLE-SAFE-02 — GOOGLE DRIVE + PERSONAL_AI ACCESS
# ============================================================

import os
from pathlib import Path

print("=" * 70)
print("KAGGLE-SAFE-02 : GOOGLE DRIVE + PERSONAL_AI")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check whether Google Drive is already mounted
# ------------------------------------------------------------

drive_candidates = [
    Path("/content/drive/MyDrive"),
    Path("/kaggle/working/drive/MyDrive"),
    Path("/kaggle/input/drive/MyDrive"),
]

print("\n[DRIVE CHECK]")

found_drive = None

for path in drive_candidates:
    if path.exists():
        found_drive = path
        print("FOUND:", path)
        break

if found_drive is None:
    print("Google Drive is not mounted in this Kaggle session.")
    print("\nIMPORTANT:")
    print("Kaggle normally does NOT provide Colab-style drive.mount().")
    print("So we will NOT assume direct Google Drive mounting.")
    print("\nKAGGLE-SAFE-02 STATUS: NEED DRIVE BRIDGE")
else:
    print("Drive root:", found_drive)

# ------------------------------------------------------------
# 2. Search common Kaggle locations for Personal_AI
# ------------------------------------------------------------

print("\n[PERSONAL_AI SEARCH]")

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/content"),
]

matches = []

for root in search_roots:
    if not root.exists():
        continue

    try:
        for p in root.rglob("Personal_AI"):
            if p.is_dir():
                matches.append(p)
    except Exception as e:
        print("Search warning:", root, repr(e))

if matches:
    for p in matches:
        print("FOUND:", p)
else:
    print("Personal_AI not found in current Kaggle filesystem.")

# ------------------------------------------------------------
# 3. Inspect Kaggle working/input directories
# ------------------------------------------------------------

print("\n[KAGGLE DIRECTORIES]")

for p in [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]:
    print(f"\n{p}")
    if p.exists():
        try:
            items = list(p.iterdir())
            for item in items[:30]:
                print("  ", item)
            if len(items) > 30:
                print("   ...", len(items) - 30, "more")
        except Exception as e:
            print("   ERROR:", repr(e))

# ------------------------------------------------------------
# 4. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if found_drive:
    print("KAGGLE-SAFE-02 STATUS: DRIVE ACCESS FOUND")
else:
    print("KAGGLE-SAFE-02 STATUS: DRIVE ACCESS NOT YET AVAILABLE")

print("=" * 70)

In [ ]:
# ============================================================
# KAGGLE-SAFE-03 — STORAGE + NETWORK BRIDGE AUDIT
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

print("=" * 70)
print("KAGGLE-SAFE-03 : STORAGE + NETWORK BRIDGE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Kaggle environment
# ------------------------------------------------------------

print("\n[KAGGLE ENVIRONMENT]")

for key in [
    "KAGGLE_KERNEL_RUN_TYPE",
    "KAGGLE_URL_BASE",
    "KAGGLE_DATA_PROXY_TOKEN",
]:
    value = os.environ.get(key)

    if value:
        # Do not print credentials/tokens
        if "TOKEN" in key:
            print(key, ": PRESENT (hidden)")
        else:
            print(key, ":", value)
    else:
        print(key, ": not set")

# ------------------------------------------------------------
# 2. Storage
# ------------------------------------------------------------

print("\n[STORAGE]")

for path in [
    "/kaggle/working",
    "/kaggle/input",
    "/kaggle/temp",
    "/content",
]:
    p = Path(path)

    if p.exists():
        print(f"{path:20} EXISTS")
    else:
        print(f"{path:20} not present")

# Disk information
print("\n[DISK SPACE]")

try:
    result = subprocess.run(
        ["df", "-h", "/kaggle/working"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print(result.stdout)
except Exception as e:
    print("Disk check failed:", repr(e))

# ------------------------------------------------------------
# 3. Network
# ------------------------------------------------------------

print("\n[NETWORK]")

try:
    result = subprocess.run(
        ["python", "-c",
         "import urllib.request; "
         "r=urllib.request.urlopen('https://huggingface.co', timeout=10); "
         "print('HTTP', r.status)"],
        capture_output=True,
        text=True,
        timeout=20
    )

    print(result.stdout.strip())

    if result.stderr:
        print("stderr:", result.stderr.strip())

except Exception as e:
    print("Internet test failed:", repr(e))

# ------------------------------------------------------------
# 4. Git
# ------------------------------------------------------------

print("\n[GIT]")

try:
    result = subprocess.run(
        ["git", "--version"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print(result.stdout.strip())
except Exception as e:
    print("Git unavailable:", repr(e))

# ------------------------------------------------------------
# 5. Kaggle API package
# ------------------------------------------------------------

print("\n[KAGGLE API PACKAGE]")

try:
    import kaggle
    print("kaggle package: OK")
except Exception as e:
    print("kaggle package:", repr(e))

# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("KAGGLE-SAFE-03 COMPLETE")
print("=" * 70)

print("""
NEXT:
We will choose the persistent-storage bridge based on this audit.

DO NOT:
- download Qwen again
- download a video model
- modify Personal_AI
- upload private credentials/tokens
""")

In [ ]:
# ============================================================
# KAGGLE/COLAB-SAFE-04A — ENVIRONMENT CHECK
# ============================================================

import os
import sys
from pathlib import Path

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

# Detect Kaggle
is_kaggle = (
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
    or Path("/kaggle/working").exists()
)

# Detect Colab
is_colab = "google.colab" in sys.modules

print("\n[KAGGLE]")
print("Detected:", is_kaggle)

print("\n[COLAB]")
print("Detected:", is_colab)

print("\n[DRIVE]")
drive_path = Path("/content/drive/MyDrive")

print("Drive path exists:", drive_path.exists())

print("\n[CURRENT DIRECTORY]")
print(Path.cwd())

print("\n" + "=" * 70)

if is_kaggle:
    print("STATUS: KAGGLE RUNTIME")
    print()
    print("Do NOT run Google Drive structure audit here.")
    print("Kaggle does not currently have your Colab Drive mount.")
    print()
    print("Next: return to the Colab Personal_AI notebook")
    print("and mount Google Drive there.")

elif is_colab:
    if drive_path.exists():
        print("STATUS: COLAB + DRIVE ALREADY MOUNTED")
        print("Personal_AI path can now be checked.")
    else:
        print("STATUS: COLAB BUT DRIVE NOT MOUNTED")
        print("Next: mount Google Drive first.")

else:
    print("STATUS: UNKNOWN ENVIRONMENT")

print("=" * 70)

In [ ]:
# ================================================================
# KAGGLE-SAFE-01 : KAGGLE API ACCESS AUDIT
# ================================================================

import os
import subprocess
from pathlib import Path

print("=" * 70)
print("KAGGLE-SAFE-01 : KAGGLE API ACCESS AUDIT")
print("=" * 70)

# ------------------------------------------------
# 1. Kaggle package
# ------------------------------------------------

try:
    import kaggle
    print("\n[KAGGLE PACKAGE]")
    print("STATUS  : OK")
    print("VERSION :", getattr(kaggle, "__version__", "unknown"))
except Exception as e:
    print("\n[KAGGLE PACKAGE]")
    print("STATUS  : FAIL")
    print("ERROR   :", repr(e))

# ------------------------------------------------
# 2. Credential presence check
#    Do NOT print credentials.
# ------------------------------------------------

print("\n[CREDENTIAL CHECK]")

credential_candidates = [
    Path.home() / ".kaggle" / "kaggle.json",
    Path("/root/.kaggle/kaggle.json"),
    Path("/content/.kaggle/kaggle.json"),
]

found = []

for p in credential_candidates:
    if p.exists():
        found.append(str(p))

if found:
    print("STATUS  : FOUND")
    for p in found:
        print("  ", p)
    print("Credential contents NOT displayed.")
else:
    print("STATUS  : NOT FOUND")
    print("No local Kaggle API credential file detected.")

# ------------------------------------------------
# 3. Environment credential presence
# ------------------------------------------------

env_keys = [
    "KAGGLE_USERNAME",
    "KAGGLE_KEY",
    "KAGGLE_API_TOKEN",
]

print("\n[ENVIRONMENT CREDENTIAL CHECK]")

for key in env_keys:
    print(f"{key:18} : {'SET' if os.environ.get(key) else 'NOT SET'}")

# ------------------------------------------------
# 4. Safe API identity test
# ------------------------------------------------

print("\n[API IDENTITY TEST]")

try:
    result = subprocess.run(
        ["kaggle", "whoami"],
        capture_output=True,
        text=True,
        timeout=30
    )

    print("Return code :", result.returncode)

    if result.stdout.strip():
        print("STDOUT:")
        print(result.stdout.strip())

    if result.stderr.strip():
        print("STDERR:")
        print(result.stderr.strip())

    if result.returncode == 0:
        print("STATUS      : PASS")
    else:
        print("STATUS      : NOT AUTHENTICATED / FAILED")

except Exception as e:
    print("STATUS      : ERROR")
    print("ERROR       :", repr(e))

print("\n" + "=" * 70)
print("KAGGLE-SAFE-01 COMPLETE")
print("=" * 70)

print("\nNO DATASET CREATED")
print("NO FILE UPLOADED")
print("NO PERSONAL_AI SOURCE MODIFIED")

In [ ]:
# ================================================================
# KAGGLE-SAFE-02 : SECURE KAGGLE AUTHENTICATION
# ================================================================

import os
from pathlib import Path
from getpass import getpass
import subprocess

print("=" * 70)
print("KAGGLE-SAFE-02 : SECURE AUTHENTICATION")
print("=" * 70)

token = getpass("Paste your NEW Kaggle API token here (hidden): ").strip()

if not token:
    raise ValueError("No token entered.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

token_path = kaggle_dir / "access_token"

token_path.write_text(token + "\n", encoding="utf-8")
os.chmod(token_path, 0o600)

# Remove token from Python variable
del token

print("\n[CREDENTIAL]")
print("Path       :", token_path)
print("Permissions:", oct(token_path.stat().st_mode & 0o777))
print("Token      : NOT DISPLAYED")

# ------------------------------------------------
# Supported authenticated API test
# ------------------------------------------------

print("\n[API TEST]")

result = subprocess.run(
    ["kaggle", "competitions", "list", "--page", "1"],
    capture_output=True,
    text=True,
    timeout=60
)

print("Return code:", result.returncode)

if result.returncode == 0:
    print("STATUS     : PASS")
    print("\nKaggle API is authenticated.")
else:
    print("STATUS     : FAIL")
    print("\nSTDOUT:")
    print(result.stdout[:2000])
    print("\nSTDERR:")
    print(result.stderr[:2000])

print("\n" + "=" * 70)
print("KAGGLE-SAFE-02 COMPLETE")
print("=" * 70)

print("\nDATASET CREATED : NO")
print("FILES UPLOADED  : NO")
print("SOURCE MODIFIED : NO")

In [ ]:
# ================================================================
# KAGGLE-SAFE-03 : CREATE + UPLOAD PERSONAL AI CORE DATASET
# ================================================================

import json
import subprocess
from pathlib import Path

print("=" * 70)
print("KAGGLE-SAFE-03 : PERSONAL AI CORE DATASET")
print("=" * 70)

CORE = Path(
    "/content/drive/MyDrive/Personal_AI/kaggle_export/personal_ai_core"
)

if not CORE.exists():
    raise FileNotFoundError(f"Core package not found: {CORE}")

files = [p for p in CORE.rglob("*") if p.is_file()]
size = sum(p.stat().st_size for p in files)

print("\n[PACKAGE]")
print("Path  :", CORE)
print("Files :", len(files))
print("Size  :", f"{size/1024/1024:.3f} MB")

# ------------------------------------------------
# Get Kaggle username safely
# ------------------------------------------------

print("\n[KAGGLE IDENTITY]")

result = subprocess.run(
    ["kaggle", "config", "view"],
    capture_output=True,
    text=True,
    timeout=30
)

print("Return code:", result.returncode)

if result.stdout.strip():
    print(result.stdout.strip())

if result.stderr.strip():
    print("STDERR:", result.stderr.strip())

# ------------------------------------------------
# Ask username explicitly
# ------------------------------------------------

username = input(
    "\nEnter your Kaggle username (NOT API token): "
).strip()

if not username:
    raise ValueError("Kaggle username is required.")

dataset_slug = "personal-ai-core"

# ------------------------------------------------
# Dataset metadata
# ------------------------------------------------

metadata = {
    "title": "Personal AI Core",
    "id": f"{username}/{dataset_slug}",
    "description": (
        "Core architecture package for the Personal AI system. "
        "Contains core contracts, memory, runtime registries, "
        "modules, generation metadata and configuration."
    ),
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ],
    "keywords": [
        "personal-ai",
        "llm",
        "ai-agent",
        "reel-generation"
    ]
}

metadata_path = CORE / "dataset-metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\n[METADATA]")
print("Created:", metadata_path)
print("Dataset :", f"{username}/{dataset_slug}")

# ------------------------------------------------
# Upload
# ------------------------------------------------

print("\n[UPLOAD]")
print("Uploading ONLY personal_ai_core...")
print("Original Personal_AI source will NOT be modified.")

upload = subprocess.run(
    [
        "kaggle",
        "datasets",
        "create",
        "-p",
        str(CORE),
        "-r",
        "skip"
    ],
    capture_output=True,
    text=True,
    timeout=300
)

print("\nReturn code:", upload.returncode)

if upload.stdout.strip():
    print("\nSTDOUT:")
    print(upload.stdout.strip())

if upload.stderr.strip():
    print("\nSTDERR:")
    print(upload.stderr.strip())

if upload.returncode == 0:
    print("\nSTATUS : PASS")
    print("personal_ai_core dataset created successfully.")
else:
    print("\nSTATUS : FAIL")
    print("Dataset creation failed. DO NOT proceed to Qwen yet.")

print("\n" + "=" * 70)
print("KAGGLE-SAFE-03 COMPLETE")
print("=" * 70)

print("\nQWEN UPLOAD : NOT PERFORMED")
print("SOURCE MODIFIED : NO")

In [ ]:
# ================================================================
# KAGGLE-SAFE-03 : UPLOAD PERSONAL AI CORE DATASET
# ================================================================

import json
import subprocess
from pathlib import Path

print("=" * 70)
print("KAGGLE-SAFE-03 : PERSONAL AI CORE DATASET UPLOAD")
print("=" * 70)

CORE = Path(
    "/content/drive/MyDrive/Personal_AI/kaggle_export/personal_ai_core"
)

assert CORE.exists(), f"Core package missing: {CORE}"

files = [p for p in CORE.rglob("*") if p.is_file()]
size = sum(p.stat().st_size for p in files)

print("\n[PACKAGE]")
print("Files :", len(files))
print("Size  :", f"{size/1024/1024:.3f} MB")

# ------------------------------------------------
# Kaggle username
# ------------------------------------------------

username = input("\nEnter your Kaggle username: ").strip()

if not username:
    raise ValueError("Kaggle username cannot be empty.")

slug = "personal-ai-core"

# ------------------------------------------------
# Create metadata
# ------------------------------------------------

metadata = {
    "title": "Personal AI Core",
    "id": f"{username}/{slug}",
    "description": (
        "Core package for the Personal AI system: "
        "module contracts, memory, runtime registries, "
        "modules, generation metadata and configuration."
    ),
    "licenses": [
        {"name": "CC0-1.0"}
    ]
}

metadata_path = CORE / "dataset-metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\n[DATASET]")
print("ID:", metadata["id"])

# ------------------------------------------------
# Upload
# ------------------------------------------------

print("\n[UPLOAD]")
print("Uploading core package only...")

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "create",
        "-p",
        str(CORE),
        "-r",
        "skip"
    ],
    capture_output=True,
    text=True,
    timeout=300
)

print("\nReturn code:", result.returncode)

if result.stdout.strip():
    print("\nSTDOUT:")
    print(result.stdout.strip())

if result.stderr.strip():
    print("\nSTDERR:")
    print(result.stderr.strip())

if result.returncode == 0:
    print("\nSTATUS : PASS")
    print("Core dataset uploaded successfully.")
else:
    print("\nSTATUS : FAIL")
    print("Do NOT upload Qwen yet.")

print("\n" + "=" * 70)
print("KAGGLE-SAFE-03 COMPLETE")
print("=" * 70)

print("QWEN UPLOAD      : NOT PERFORMED")
print("SOURCE MODIFIED  : NO")

In [ ]:
# ================================================================
# KAGGLE-SAFE-03 : UPLOAD PERSONAL_AI_CORE DATASET
# ================================================================

from pathlib import Path
import subprocess
import json

CORE = Path("/content/drive/MyDrive/Personal_AI/kaggle_export/personal_ai_core")

DATASET_DIR = Path("/content/kaggle_dataset_personal_ai_core")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Fresh staging copy
import shutil

STAGED_CORE = DATASET_DIR / "personal_ai_core"

if STAGED_CORE.exists():
    shutil.rmtree(STAGED_CORE)

shutil.copytree(CORE, STAGED_CORE)

# Kaggle dataset metadata
metadata = {
    "title": "Personal AI Core",
    "id": "mydivvacc/personal-ai-core",
    "description": "Minimal Personal AI core runtime package for Kaggle execution.",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ],
    "is_private": True
}

with open(DATASET_DIR / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("=" * 70)
print("KAGGLE-SAFE-03 : UPLOAD PERSONAL_AI_CORE")
print("=" * 70)

print("\n[STAGING CHECK]")

files = [
    p for p in STAGED_CORE.rglob("*")
    if p.is_file()
    and "__pycache__" not in p.parts
    and p.suffix != ".pyc"
]

size_mb = sum(p.stat().st_size for p in files) / (1024 * 1024)

print(f"Files : {len(files)}")
print(f"Size  : {size_mb:.3f} MB")

assert len(files) == 29
assert size_mb < 1.0

print("\nSTAGING : PASS")

print("\n[UPLOAD]")
print("Creating private Kaggle dataset...")

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "create",
        "-p",
        str(DATASET_DIR),
        "--dir-mode",
        "zip"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n[STDERR]")
    print(result.stderr)

print("\nReturn code:", result.returncode)

if result.returncode == 0:
    print("\nSTATUS : PASS")
else:
    print("\nSTATUS : CHECK REQUIRED")

In [ ]:
# ================================================================
# KAGGLE-SAFE-03A : DRIVE → LOCAL STAGING → KAGGLE CORE UPLOAD
# ================================================================

from google.colab import drive
from pathlib import Path
import shutil, subprocess, json

print("=" * 70)
print("KAGGLE-SAFE-03A : PERSONAL_AI_CORE UPLOAD")
print("=" * 70)

# 1. Mount Drive
drive.mount("/content/drive", force_remount=False)

MASTER = Path("/content/drive/MyDrive/Personal_AI")

SOURCE_DIRS = [
    "01_core",
    "05_memory",
    "05_runtime",
    "06_modules",
    "08_generation",
    "10_config",
]

# 2. Verify master
assert MASTER.exists(), f"Personal_AI missing: {MASTER}"

print("\n[SOURCE CHECK]")
for d in SOURCE_DIRS:
    p = MASTER / d
    print(f"{'OK':8s} {d}")
    assert p.exists(), f"Missing source: {p}"

# 3. Temporary LOCAL staging — NOT Drive
STAGE_ROOT = Path("/content/kaggle_personal_ai_core")
STAGE_CORE = STAGE_ROOT / "personal_ai_core"

if STAGE_ROOT.exists():
    shutil.rmtree(STAGE_ROOT)

STAGE_CORE.mkdir(parents=True)

print("\n[LOCAL STAGING]")

for d in SOURCE_DIRS:
    src = MASTER / d
    dst = STAGE_CORE / d
    shutil.copytree(src, dst)

# Remove Python cache/compiled files from staging
for p in list(STAGE_CORE.rglob("__pycache__")):
    if p.is_dir():
        shutil.rmtree(p)

for p in list(STAGE_CORE.rglob("*.pyc")):
    if p.is_file():
        p.unlink()

# 4. Validate staged package
files = [
    p for p in STAGE_CORE.rglob("*")
    if p.is_file()
]

size_mb = sum(p.stat().st_size for p in files) / (1024 * 1024)

print(f"Files : {len(files)}")
print(f"Size  : {size_mb:.3f} MB")

assert len(files) == 29, f"Expected 29 files, found {len(files)}"
assert size_mb < 1.0

print("STAGING : PASS")

# 5. Kaggle metadata
metadata = {
    "title": "Personal AI Core",
    "id": "mydivvacc/personal-ai-core",
    "description": "Personal AI core runtime package.",
    "licenses": [{"name": "CC0-1.0"}],
    "is_private": True
}

with open(STAGE_ROOT / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# 6. Verify Kaggle CLI
print("\n[KAGGLE CLI CHECK]")

check = subprocess.run(
    ["kaggle", "--version"],
    capture_output=True,
    text=True
)

print(check.stdout.strip())

assert check.returncode == 0

# 7. Upload
print("\n[UPLOAD]")
print("Uploading ONLY personal_ai_core...")
print("Qwen is NOT being uploaded.")

result = subprocess.run(
    [
        "kaggle",
        "datasets",
        "create",
        "-p",
        str(STAGE_ROOT),
        "--dir-mode",
        "zip"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n[STDERR]")
    print(result.stderr)

print("\nReturn code:", result.returncode)

print("=" * 70)

if result.returncode == 0:
    print("STATUS : PASS")
else:
    print("STATUS : CHECK REQUIRED")

In [ ]:
# KAGGLE → GOOGLE DRIVE TEST

!pip install -q PyDrive2

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

gauth = GoogleAuth()
gauth.LocalWebserverAuth()

drive = GoogleDrive(gauth)

print("Google Drive connected successfully!")

# Find Personal_AI folder
results = drive.ListFile({
    "q": "title='Personal_AI' and trashed=false"
}).GetList()

print("\nPersonal_AI folders found:", len(results))

for folder in results:
    print("Name:", folder["title"])
    print("ID:", folder["id"])

In [1]:
# ============================================================
# KAGGLE-SAFE-04 : PERSONAL AI CORE DATASET VERIFICATION
# ============================================================

from pathlib import Path
import hashlib

ROOT = Path("/kaggle/input/personal-ai-core")

print("=" * 65)
print("📦 PERSONAL AI CORE — KAGGLE DATASET AUDIT")
print("=" * 65)

print("\nDataset path:")
print(ROOT)

print("\nExists:", ROOT.exists())

if not ROOT.exists():
    raise FileNotFoundError(
        f"❌ Dataset not found at:\n{ROOT}\n\n"
        "Check Kaggle → Input → Datasets."
    )

files = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file()
)

print("📄 File count:", len(files))

print("\n📋 Files:")
for i, p in enumerate(files, 1):
    size_kb = p.stat().st_size / 1024
    print(f"{i:02d}. {p.relative_to(ROOT)} ({size_kb:.1f} KB)")

# ------------------------------------------------------------
# Expected critical files
# ------------------------------------------------------------

required = [
    "01_core/module_contract.py",

    "05_memory/memory_db.json",
    "05_memory/personal_memory.json",

    "05_runtime/knowledge_registry.json",
    "05_runtime/module_registry.json",

    "06_modules/video_reels/module.json",
    "06_modules/video_reels/config/default.json",
    "06_modules/video_reels/directors/reel_director.py",

    "08_generation/job_schema.json",

    "10_config/system_config.json",
    "10_config/model_registry.json",
    "10_config/module_registry.json",
    "10_config/hardware_profile.json",
]

print("\n🔎 Critical file check:")

missing = []

for rel in required:
    path = ROOT / rel

    if path.exists():
        print("✅", rel)
    else:
        print("❌", rel)
        missing.append(rel)

# ------------------------------------------------------------
# Python / JSON sanity check
# ------------------------------------------------------------

print("\n🧪 Basic Python/JSON validation:")

python_errors = []
json_errors = []

for p in files:

    if p.suffix == ".py":
        try:
            compile(
                p.read_text(encoding="utf-8"),
                str(p),
                "exec"
            )
        except Exception as e:
            python_errors.append((str(p.relative_to(ROOT)), str(e)))

    elif p.suffix == ".json":
        try:
            import json
            json.loads(p.read_text(encoding="utf-8"))
        except Exception as e:
            json_errors.append((str(p.relative_to(ROOT)), str(e)))

print("Python errors:", len(python_errors))
print("JSON errors:", len(json_errors))

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n" + "=" * 65)

if (
    len(files) == 29
    and not missing
    and not python_errors
    and not json_errors
):
    print("🎉 PASS — PERSONAL AI CORE DATASET IS HEALTHY")
else:
    print("⚠️ CHECK REQUIRED")

    if missing:
        print("\nMissing files:")
        for x in missing:
            print(" -", x)

    if python_errors:
        print("\nPython errors:")
        for x in python_errors:
            print(" -", x)

    if json_errors:
        print("\nJSON errors:")
        for x in json_errors:
            print(" -", x)

print("=" * 65)

📦 PERSONAL AI CORE — KAGGLE DATASET AUDIT

Dataset path:
/kaggle/input/personal-ai-core

Exists: False


FileNotFoundError: ❌ Dataset not found at:
/kaggle/input/personal-ai-core

Check Kaggle → Input → Datasets.

In [2]:
# ============================================================
# KAGGLE-SAFE-04.1 : DISCOVER ACTUAL INPUT DATASET PATH
# ============================================================

from pathlib import Path

print("=" * 65)
print("🔎 KAGGLE INPUT DATASET DISCOVERY")
print("=" * 65)

INPUT = Path("/kaggle/input")

print("\n/kaggle/input exists:", INPUT.exists())

if not INPUT.exists():
    raise RuntimeError(
        "❌ /kaggle/input itself is unavailable. "
        "This does not look like a normal Kaggle Notebook runtime."
    )

print("\n📂 Contents of /kaggle/input:")

items = sorted(INPUT.iterdir())

if not items:
    print("❌ No datasets/inputs are currently attached.")
else:
    for item in items:
        print(
            f"{'📁' if item.is_dir() else '📄'} "
            f"{item.name}"
        )

print("\n" + "=" * 65)

# Search for our dataset by name/content
matches = []

for p in INPUT.rglob("*"):
    if p.is_file():
        if (
            "personal" in p.name.lower()
            or "ai" in p.name.lower()
            or "module_contract.py" in p.name.lower()
        ):
            matches.append(p)

print("🔍 Relevant files found:", len(matches))

for p in matches[:50]:
    print(" -", p)

print("\n" + "=" * 65)
print("✅ DISCOVERY COMPLETE")
print("=" * 65)

🔎 KAGGLE INPUT DATASET DISCOVERY

/kaggle/input exists: True

📂 Contents of /kaggle/input:
📁 datasets

🔍 Relevant files found: 2
 - /kaggle/input/datasets/vikaschandoliya/personal-ai-core/01_core/module_contract.py
 - /kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_memory/personal_memory.json

✅ DISCOVERY COMPLETE


In [3]:
# ============================================================
# KAGGLE-SAFE-04.2 : VERIFY ATTACHED PERSONAL AI CORE
# ============================================================

from pathlib import Path
import json

ROOT = Path(
    "/kaggle/input/datasets/vikaschandoliya/personal-ai-core"
)

print("=" * 65)
print("📦 PERSONAL AI CORE — ATTACHED DATASET VERIFICATION")
print("=" * 65)

print("\nDataset path:")
print(ROOT)

if not ROOT.exists():
    raise FileNotFoundError(f"❌ Dataset not found:\n{ROOT}")

files = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file()
)

print(f"\n✅ Dataset found")
print(f"📄 Total files: {len(files)}")

print("\n📋 Files:")
for i, p in enumerate(files, 1):
    print(f"{i:02d}. {p.relative_to(ROOT)}")

# ------------------------------------------------------------
# Critical files
# ------------------------------------------------------------

required = [
    "01_core/module_contract.py",

    "05_memory/memory_db.json",
    "05_memory/personal_memory.json",

    "05_runtime/knowledge_registry.json",
    "05_runtime/module_registry.json",

    "06_modules/video_reels/module.json",
    "06_modules/video_reels/config/default.json",
    "06_modules/video_reels/directors/reel_director.py",

    "08_generation/job_schema.json",

    "10_config/system_config.json",
    "10_config/model_registry.json",
    "10_config/module_registry.json",
    "10_config/hardware_profile.json",
]

print("\n🔎 Critical file verification:")

missing = []

for rel in required:
    if (ROOT / rel).exists():
        print("✅", rel)
    else:
        print("❌", rel)
        missing.append(rel)

# ------------------------------------------------------------
# Validate Python + JSON
# ------------------------------------------------------------

python_errors = []
json_errors = []

for p in files:

    if p.suffix == ".py":
        try:
            compile(
                p.read_text(encoding="utf-8"),
                str(p),
                "exec"
            )
        except Exception as e:
            python_errors.append(
                (str(p.relative_to(ROOT)), str(e))
            )

    elif p.suffix == ".json":
        try:
            json.loads(
                p.read_text(encoding="utf-8")
            )
        except Exception as e:
            json_errors.append(
                (str(p.relative_to(ROOT)), str(e))
            )

print("\n🧪 Validation:")
print("Python errors:", len(python_errors))
print("JSON errors:", len(json_errors))

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 65)

if (
    len(files) == 29
    and not missing
    and not python_errors
    and not json_errors
):
    print("🎉 PASS — PERSONAL AI CORE IS FULLY VERIFIED")
else:
    print("⚠️ CHECK REQUIRED")

    if missing:
        print("\nMissing:")
        for x in missing:
            print(" -", x)

    if python_errors:
        print("\nPython errors:")
        for x in python_errors:
            print(" -", x)

    if json_errors:
        print("\nJSON errors:")
        for x in json_errors:
            print(" -", x)

print("=" * 65)

📦 PERSONAL AI CORE — ATTACHED DATASET VERIFICATION

Dataset path:
/kaggle/input/datasets/vikaschandoliya/personal-ai-core

✅ Dataset found
📄 Total files: 29

📋 Files:
01. 01_core/module_contract.py
02. 05_memory/memory_db.json
03. 05_memory/personal_memory.json
04. 05_runtime/knowledge_registry.json
05. 05_runtime/module_registry.json
06. 06_modules/video_reels/README.md
07. 06_modules/video_reels/config/default.json
08. 06_modules/video_reels/directors/reel_director.py
09. 06_modules/video_reels/module.json
10. 06_modules/video_reels/prompts/reel_director.txt
11. 06_modules/video_reels/tests/foundation_test.json
12. 08_generation/job_schema.json
13. 08_generation/jobs/20260823_211818_77d9d84f.json
14. 08_generation/outputs/reels/safe31_2_reel_plan_20260910_191409.json
15. 08_generation/products/product_6be5e33e7015.json
16. 08_generation/products/product_afe698c4ff67.json
17. 08_generation/products/product_cac27378a342.json
18. 10_config/generation_engine_status.json
19. 10_config/har

In [4]:
# ============================================================
# KAGGLE-SAFE-05 : GPU + RUNTIME AUDIT
# ============================================================

import os
import shutil
import platform
import torch

print("=" * 65)
print("🚀 PERSONAL AI — KAGGLE GPU/RUNTIME AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Python / OS
# ------------------------------------------------------------

print("\n🐍 Python:")
print(platform.python_version())

print("\n💻 Platform:")
print(platform.platform())

# ------------------------------------------------------------
# 2. PyTorch
# ------------------------------------------------------------

print("\n🔥 PyTorch:")
print(torch.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

# ------------------------------------------------------------
# 3. GPUs
# ------------------------------------------------------------

gpu_count = torch.cuda.device_count()

print("\n🎮 GPU count:", gpu_count)

for i in range(gpu_count):
    gpu = torch.cuda.get_device_properties(i)

    total_gb = gpu.total_memory / (1024**3)
    free_gb, used_gb = torch.cuda.mem_get_info(i)
    free_gb /= (1024**3)
    used_gb /= (1024**3)

    print(f"\nGPU {i}")
    print("Name:", gpu.name)
    print(f"VRAM total: {total_gb:.2f} GB")
    print(f"VRAM free : {free_gb:.2f} GB")
    print(f"VRAM used : {used_gb:.2f} GB")

# ------------------------------------------------------------
# 4. Disk
# ------------------------------------------------------------

print("\n💾 Disk:")

for path in ["/kaggle/input", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(path)

    print(f"\n{path}")
    print(f"Total: {total / (1024**3):.2f} GB")
    print(f"Used : {used / (1024**3):.2f} GB")
    print(f"Free : {free / (1024**3):.2f} GB")

# ------------------------------------------------------------
# 5. Personal AI Dataset
# ------------------------------------------------------------

ROOT = "/kaggle/input/datasets/vikaschandoliya/personal-ai-core"

print("\n📦 Personal AI Core:")
print("Path:", ROOT)
print("Exists:", os.path.exists(ROOT))

# ------------------------------------------------------------
# 6. Environment
# ------------------------------------------------------------

print("\n🌐 Kaggle environment:")

print("KAGGLE_KERNEL_RUN_TYPE:",
      os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

print("KAGGLE_URL_BASE:",
      os.environ.get("KAGGLE_URL_BASE"))

# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("\n" + "=" * 65)

if torch.cuda.is_available() and gpu_count >= 1 and os.path.exists(ROOT):
    print("🎉 PASS — KAGGLE GPU + PERSONAL AI CORE READY")
else:
    print("⚠️ CHECK REQUIRED")

print("=" * 65)

🚀 PERSONAL AI — KAGGLE GPU/RUNTIME AUDIT

🐍 Python:
3.12.13

💻 Platform:
Linux-6.12.90+-x86_64-with-glibc2.35

🔥 PyTorch:
2.10.0+cu128

CUDA available: True
CUDA version: 12.8

🎮 GPU count: 2

GPU 0
Name: Tesla T4
VRAM total: 14.56 GB
VRAM free : 14.46 GB
VRAM used : 14.56 GB

GPU 1
Name: Tesla T4
VRAM total: 14.56 GB
VRAM free : 14.46 GB
VRAM used : 14.56 GB

💾 Disk:

/kaggle/input
Total: 19.52 GB
Used : 0.00 GB
Free : 19.50 GB

/kaggle/working
Total: 19.52 GB
Used : 0.00 GB
Free : 19.50 GB

📦 Personal AI Core:
Path: /kaggle/input/datasets/vikaschandoliya/personal-ai-core
Exists: True

🌐 Kaggle environment:
KAGGLE_KERNEL_RUN_TYPE: Interactive
KAGGLE_URL_BASE: https://www.kaggle.com

🎉 PASS — KAGGLE GPU + PERSONAL AI CORE READY


In [5]:
# ============================================================
# KAGGLE-SAFE-06 : QWEN INPUT DISCOVERY
# ============================================================

from pathlib import Path
import os

print("=" * 65)
print("🔎 PERSONAL AI — QWEN INPUT DISCOVERY")
print("=" * 65)

INPUT_ROOT = Path("/kaggle/input")

print("\n📂 /kaggle/input contents:\n")

for p in sorted(INPUT_ROOT.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"FILE  | {size_mb:8.2f} MB | {p}")

print("\n" + "=" * 65)

# Look specifically for Qwen/model-related files
patterns = [
    "*Qwen*",
    "*qwen*",
    "*model.safetensors",
    "*config.json",
    "*tokenizer.json",
]

matches = set()

for pattern in patterns:
    for p in INPUT_ROOT.rglob(pattern):
        if p.is_file():
            matches.add(p)

print("\n🤖 QWEN / MODEL RELATED FILES:")

if matches:
    for p in sorted(matches):
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"{size_mb:8.2f} MB | {p}")
else:
    print("No Qwen/model files currently attached.")

print("\n" + "=" * 65)
print("📌 NEXT ACTION:")
print("If no Qwen files appear, we will attach/upload the Qwen dataset next.")
print("=" * 65)

🔎 PERSONAL AI — QWEN INPUT DISCOVERY

📂 /kaggle/input contents:

FILE  |     0.00 MB | /kaggle/input/datasets/vikaschandoliya/personal-ai-core/01_core/module_contract.py
FILE  |     0.00 MB | /kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_memory/memory_db.json
FILE  |     0.00 MB | /kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_memory/personal_memory.json
FILE  |     0.00 MB | /kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_runtime/knowledge_registry.json
FILE  |     0.00 MB | /kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_runtime/module_registry.json
FILE  |     0.00 MB | /kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/README.md
FILE  |     0.00 MB | /kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/config/default.json
FILE  |     0.01 MB | /kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/directors/reel_director.py
FILE  |     0.00 MB | /kaggle/input

In [6]:
# ============================================================
# KAGGLE-SAFE-07 : QWEN DATASET UPLOAD READINESS
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path

print("=" * 65)
print("🤖 PERSONAL AI — QWEN UPLOAD READINESS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Kaggle authentication status
# ------------------------------------------------------------

print("\n🔐 Kaggle API:")

env_token = os.environ.get("KAGGLE_API_TOKEN")

print("KAGGLE_API_TOKEN present:", bool(env_token))

result = subprocess.run(
    ["kaggle", "datasets", "list", "--page", "1"],
    capture_output=True,
    text=True
)

print("API test return code:", result.returncode)

if result.returncode == 0:
    print("API access: PASS")
else:
    print("API access: CHECK")
    print(result.stderr[:500])

# ------------------------------------------------------------
# 2. Working disk
# ------------------------------------------------------------

print("\n💾 Kaggle working disk:")

total, used, free = shutil.disk_usage("/kaggle/working")

print(f"Total: {total / (1024**3):.2f} GB")
print(f"Used : {used / (1024**3):.2f} GB")
print(f"Free : {free / (1024**3):.2f} GB")

# ------------------------------------------------------------
# 3. Current Personal AI Core
# ------------------------------------------------------------

CORE = Path(
    "/kaggle/input/datasets/vikaschandoliya/personal-ai-core"
)

print("\n📦 Personal AI Core:")
print("Exists:", CORE.exists())

# ------------------------------------------------------------
# 4. Check whether Qwen is already attached
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")

qwen_files = []

for p in INPUT_ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()

        if (
            "qwen" in str(p).lower()
            or name == "model.safetensors"
        ):
            qwen_files.append(p)

print("\n🤖 Qwen currently attached:")

if qwen_files:
    for p in qwen_files:
        size = p.stat().st_size / (1024**3)
        print(f"{size:.3f} GB | {p}")
else:
    print("NO — Qwen is not attached.")

# ------------------------------------------------------------
# 5. Check expected model size requirement
# ------------------------------------------------------------

expected_qwen_gb = 2.886

print("\n📊 Qwen package requirement:")
print(f"Expected Qwen package: ~{expected_qwen_gb:.3f} GB")
print(f"Kaggle working free   : {free / (1024**3):.2f} GB")

print("\n" + "=" * 65)

if result.returncode == 0 and CORE.exists():
    print("🎉 PASS — READY FOR QWEN DATASET TRANSFER")
else:
    print("⚠️ CHECK REQUIRED")

print("=" * 65)

🤖 PERSONAL AI — QWEN UPLOAD READINESS

🔐 Kaggle API:
KAGGLE_API_TOKEN present: False
API test return code: 0
API access: PASS

💾 Kaggle working disk:
Total: 19.52 GB
Used : 0.00 GB
Free : 19.50 GB

📦 Personal AI Core:
Exists: True

🤖 Qwen currently attached:
NO — Qwen is not attached.

📊 Qwen package requirement:
Expected Qwen package: ~2.886 GB
Kaggle working free   : 19.50 GB

🎉 PASS — READY FOR QWEN DATASET TRANSFER


In [7]:
# ============================================================
# KAGGLE-SAFE-08.1 : KAGGLE AUTH DIAGNOSTIC
# ============================================================

import os
import subprocess
from pathlib import Path

print("=" * 70)
print("🔐 KAGGLE API AUTHENTICATION DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------------------
# 1. Environment variables — SAFE CHECK
# ------------------------------------------------------------

print("\n1️⃣ Environment authentication:")

for key in [
    "KAGGLE_API_TOKEN",
    "KAGGLE_USERNAME",
    "KAGGLE_KEY"
]:
    value = os.environ.get(key)

    if value:
        if key == "KAGGLE_API_TOKEN":
            print(f"{key}: PRESENT (hidden)")
        else:
            print(f"{key}: {value}")
    else:
        print(f"{key}: NOT SET")

# ------------------------------------------------------------
# 2. Kaggle config
# ------------------------------------------------------------

print("\n2️⃣ Kaggle config:")

config = subprocess.run(
    ["kaggle", "config", "view"],
    capture_output=True,
    text=True
)

print("Return code:", config.returncode)

if config.stdout:
    print(config.stdout)

if config.stderr:
    print("STDERR:")
    print(config.stderr[:1000])

# ------------------------------------------------------------
# 3. Kaggle CLI version
# ------------------------------------------------------------

print("\n3️⃣ Kaggle CLI:")

version = subprocess.run(
    ["kaggle", "--version"],
    capture_output=True,
    text=True
)

print("Return code:", version.returncode)
print(version.stdout or version.stderr)

# ------------------------------------------------------------
# 4. API connectivity test
# ------------------------------------------------------------

print("\n4️⃣ Kaggle API test:")

test = subprocess.run(
    ["kaggle", "datasets", "list", "--page", "1"],
    capture_output=True,
    text=True
)

print("Return code:", test.returncode)

if test.stdout:
    print("\nSTDOUT:")
    print(test.stdout[:1500])

if test.stderr:
    print("\nSTDERR:")
    print(test.stderr[:1500])

# ------------------------------------------------------------
# 5. Credential files — existence only
# ------------------------------------------------------------

print("\n5️⃣ Credential file check:")

credential_paths = [
    Path("/root/.kaggle/kaggle.json"),
    Path("/root/.kaggle/access_token"),
    Path.home() / ".kaggle" / "kaggle.json",
    Path.home() / ".kaggle" / "access_token",
]

for p in credential_paths:
    print(f"{p}: {'EXISTS' if p.exists() else 'NOT FOUND'}")

print("\n" + "=" * 70)
print("📌 DIAGNOSTIC COMPLETE")
print("No credential/token contents were printed.")
print("=" * 70)

🔐 KAGGLE API AUTHENTICATION DIAGNOSTIC

1️⃣ Environment authentication:
KAGGLE_API_TOKEN: NOT SET
KAGGLE_USERNAME: NOT SET
KAGGLE_KEY: NOT SET

2️⃣ Kaggle config:
Return code: 0
Configuration values from /root/.config/kaggle
- username: vikaschandoliya
- auth_method: ACCESS_TOKEN
- path: None
- proxy: None
- competition: None


3️⃣ Kaggle CLI:
Return code: 0
Kaggle CLI 2.0.2


4️⃣ Kaggle API test:
Return code: 0

STDOUT:
ref                                                         title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
datascikhan/e-commerce-sales-and-customer-analytics         E-Commerce Sales Analytics Dataset                    29815424  2026-08-25 08:00:46.013000           5224        119       

In [12]:
# ============================================================
# KAGGLE-SAFE-09 : VERIFY PERSONAL AI DATASETS
# ============================================================

from pathlib import Path

print("=" * 70)
print("🔍 PERSONAL AI DATASET DISCOVERY")
print("=" * 70)

INPUT = Path("/kaggle/input")

print("\n📂 /kaggle/input contents:\n")

for p in sorted(INPUT.rglob("*")):
    if p.is_file():
        print(p)

print("\n" + "=" * 70)
print("🔎 DATASET ROOTS")
print("=" * 70)

roots = {}

for p in INPUT.rglob("*"):
    if p.is_dir():
        name = p.name.lower()

        if "personal-ai-core" in name:
            roots["core"] = p

        if "personal-ai-qwen" in name:
            roots["qwen"] = p

print("\nCore:", roots.get("core", "NOT FOUND"))
print("Qwen:", roots.get("qwen", "NOT FOUND"))

print("\n" + "=" * 70)

if "core" in roots and "qwen" in roots:
    print("🎉 PASS — BOTH PERSONAL AI DATASETS FOUND")
else:
    print("⚠️ STOP — ONE OR BOTH DATASETS ARE NOT ATTACHED")

print("=" * 70)

🔍 PERSONAL AI DATASET DISCOVERY

📂 /kaggle/input contents:

/kaggle/input/datasets/vikaschandoliya/personal-ai-core/01_core/module_contract.py
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_memory/memory_db.json
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_memory/personal_memory.json
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_runtime/knowledge_registry.json
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/05_runtime/module_registry.json
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/README.md
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/config/default.json
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/directors/reel_director.py
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/module.json
/kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/prompts/reel_director.txt
/kaggle/input/datase

In [13]:
# ============================================================
# KAGGLE-SAFE-10 : QWEN 2.5-1.5B GPU LOAD TEST
# ============================================================

import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 70)
print("🧠 QWEN 2.5-1.5B → GPU LOAD TEST")
print("=" * 70)

QWEN = Path(
    "/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen"
)

print("\n📂 Model path:", QWEN)
assert QWEN.exists(), "Qwen dataset not found."

print("\n🎮 CUDA:")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "CUDA GPU is not available."

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\n📦 Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    str(QWEN),
    local_files_only=True
)

print("✅ Tokenizer loaded")

print("\n🧠 Loading Qwen model...")

model = AutoModelForCausalLM.from_pretrained(
    str(QWEN),
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("✅ Qwen model loaded")

print("\n🔎 Model device map:")
print(getattr(model, "hf_device_map", "not available"))

print("\n🧪 Running inference test...")

prompt = "Write one short funny Hinglish line about a cat cleaning a messy room."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Put inputs on the model's first device
input_device = next(model.parameters()).device
inputs = {k: v.to(input_device) for k, v in inputs.items()}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("\n📝 MODEL OUTPUT:")
print("-" * 70)
print(text)
print("-" * 70)

print("\n" + "=" * 70)
print("🎉 PASS — QWEN IS RUNNING ON KAGGLE GPU")
print("=" * 70)

🧠 QWEN 2.5-1.5B → GPU LOAD TEST

📂 Model path: /kaggle/input/datasets/vikaschandoliya/personal-ai-qwen

🎮 CUDA:
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

📦 Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Tokenizer loaded

🧠 Loading Qwen model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Qwen model loaded

🔎 Model device map:
{'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1}

🧪 Running inference test...

📝 MODEL OUTPUT:
----------------------------------------------------------------------
Write one short funny Hinglish line about a cat cleaning a messy room. "Why did the cat clean its mess? Because it wanted to find its own kitty litter!" 

N

In [14]:
# ============================================================
# KAGGLE-SAFE-11 : QWEN → PERSONAL AI REEL DIRECTOR
# ============================================================

import json
import re
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 70)
print("🎬 PERSONAL AI — REEL DIRECTOR INTEGRATION TEST")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

QWEN = Path(
    "/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen"
)

CORE = Path(
    "/kaggle/input/datasets/vikaschandoliya/personal-ai-core"
)

DIRECTOR = CORE / (
    "06_modules/video_reels/directors/reel_director.py"
)

assert QWEN.exists(), "Qwen dataset missing."
assert CORE.exists(), "Core dataset missing."
assert DIRECTOR.exists(), "Reel Director missing."

print("\n✅ Qwen:", QWEN)
print("✅ Core:", CORE)
print("✅ Director:", DIRECTOR)

# ------------------------------------------------------------
# LOAD QWEN
# ------------------------------------------------------------

print("\n🧠 Loading Qwen...")

tokenizer = AutoTokenizer.from_pretrained(
    str(QWEN),
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    str(QWEN),
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("✅ Qwen loaded")

# ------------------------------------------------------------
# REEL REQUEST
# ------------------------------------------------------------

user_request = """
Create a funny 20-second Instagram Reel.

Concept:
A realistic orange cat lives in an Indian middle-class home.
The house is messy and the cat is angry because everyone expects
the cat to clean it.

The cat should speak funny natural Hinglish.
The Reel should have a strong hook, escalating comedy and a punchline.

Format: 9:16
Duration: 20 seconds
Style: realistic, cinematic, funny.
"""

# ------------------------------------------------------------
# STRICT DIRECTOR PROMPT
# ------------------------------------------------------------

system_prompt = """
You are the Reel Director of a Personal AI system.

Return ONLY valid JSON.
No markdown.
No explanation.
No commentary before or after JSON.

Create exactly this schema:

{
  "title": "string",
  "hook": "string",
  "language": "Hinglish",
  "duration_seconds": 20,
  "format": "9:16",
  "style": ["realistic", "cinematic", "funny"],
  "character": {
    "name": "string",
    "appearance": "string",
    "personality": "string"
  },
  "scenes": [
    {
      "scene_number": 1,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 2,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 3,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 4,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 5,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    }
  ]
}

Rules:
- Exactly 5 scenes.
- Every scene is exactly 4 seconds.
- Total duration = 20 seconds.
- Dialogue must be natural Hinglish.
- Make the comedy escalate.
- Final scene must contain the punchline.
- Keep the same orange cat in every scene.
- Do not invent extra fields.
"""

prompt = system_prompt + "\n\nUSER REQUEST:\n" + user_request

# ------------------------------------------------------------
# GENERATE
# ------------------------------------------------------------

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

device = next(model.parameters()).device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

print("\n🎬 Generating Reel plan...")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.35,
        top_p=0.85,
        repetition_penalty=1.05
    )

raw = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("\n📄 RAW MODEL OUTPUT:")
print("-" * 70)
print(raw)
print("-" * 70)

# ------------------------------------------------------------
# EXTRACT JSON
# ------------------------------------------------------------

match = re.search(
    r"\{.*\}",
    raw,
    flags=re.DOTALL
)

assert match, "No JSON object found in model output."

json_text = match.group(0)

plan = json.loads(json_text)

print("\n✅ JSON parsing: PASS")

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

required_top = [
    "title",
    "hook",
    "language",
    "duration_seconds",
    "format",
    "style",
    "character",
    "scenes"
]

for key in required_top:
    assert key in plan, f"Missing top-level field: {key}"

assert plan["language"] == "Hinglish"
assert plan["duration_seconds"] == 20
assert plan["format"] == "9:16"

assert isinstance(plan["scenes"], list)
assert len(plan["scenes"]) == 5

for i, scene in enumerate(plan["scenes"], start=1):

    assert scene["scene_number"] == i
    assert scene["duration_seconds"] == 4

    for key in [
        "visual_prompt",
        "dialogue",
        "action"
    ]:
        assert key in scene
        assert isinstance(scene[key], str)
        assert scene[key].strip()

assert sum(
    s["duration_seconds"]
    for s in plan["scenes"]
) == 20

# ------------------------------------------------------------
# SAVE TEST RESULT
# ------------------------------------------------------------

OUT = Path("/kaggle/working/reel_director_test.json")

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(
        plan,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n💾 Saved:")
print(OUT)

# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 KAGGLE-SAFE-11 PASS")
print("=" * 70)

print("\n🎬 TITLE:", plan["title"])
print("🪝 HOOK:", plan["hook"])
print("⏱️ Duration:", plan["duration_seconds"], "seconds")
print("📱 Format:", plan["format"])

print("\n🎭 SCENES:")

for scene in plan["scenes"]:
    print(
        f"\nScene {scene['scene_number']} "
        f"({scene['duration_seconds']}s)"
    )
    print("Dialogue:", scene["dialogue"])
    print("Action:", scene["action"])

print("\n" + "=" * 70)

🎬 PERSONAL AI — REEL DIRECTOR INTEGRATION TEST

✅ Qwen: /kaggle/input/datasets/vikaschandoliya/personal-ai-qwen
✅ Core: /kaggle/input/datasets/vikaschandoliya/personal-ai-core
✅ Director: /kaggle/input/datasets/vikaschandoliya/personal-ai-core/06_modules/video_reels/directors/reel_director.py

🧠 Loading Qwen...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Qwen loaded

🎬 Generating Reel plan...

📄 RAW MODEL OUTPUT:
----------------------------------------------------------------------
Character: The cat

Scenes:
1. The cat is sitting on the floor, looking at the mess.
2. The cat starts cleaning, but fails miserably.
3. The cat gets frustrated and yells at everyone.
4. The cat tries to clean again, but ends up making the mess worse.
5. The cat finally gives up and starts playing with toys instead.

{
  "title": "Orange Cat's Messy Home",
  "hook": "The cat is so angry!",
  "language": "Hinglish",
  "duration_seconds": 20,
  "format": "9:16",
  "style": ["realistic", "cinematic", "funny"],
  "character": {
    "name": "Orange Cat",
    "appearance": "orange cat sitting on the floor with a messy house in the background",
    "personality": "angry, frustrated, playful"
  },
  "scenes": [
    {
      "scene_number": 1,
      "duration_seconds": 4,
      "visual_prompt": "Orange Cat sits on the floor, looking at the mess.",
      "dialogue":

In [15]:
# ============================================================
# KAGGLE-SAFE-11.1 : REEL DIRECTOR QUALITY LOCK
# ============================================================

import json
import re
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 70)
print("🎬 REEL DIRECTOR — QUALITY LOCK TEST")
print("=" * 70)

QWEN = Path("/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen")

tokenizer = AutoTokenizer.from_pretrained(
    str(QWEN),
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    str(QWEN),
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

# ------------------------------------------------------------
# STRICT DIRECTOR INSTRUCTION
# ------------------------------------------------------------

prompt = r"""
You are an expert viral Instagram Reel Director.

Your output MUST contain ONLY one valid JSON object.
Do NOT write anything before or after the JSON.
Do NOT use markdown.

Create a 20-second funny viral Reel.

CONCEPT:
A realistic orange cat lives in an Indian middle-class home.
The house is extremely messy.
Everyone expects the cat to clean it.
The cat becomes increasingly frustrated.

LANGUAGE:
Natural Indian Hinglish.
Dialogue should sound like something an Indian person would
actually say casually.
Use Hindi + English naturally.
Do NOT write formal Hindi.
Do NOT make the dialogue mostly English.

COMEDY:
Scene 1 = strong curiosity hook.
Scene 2 = situation becomes more ridiculous.
Scene 3 = cat becomes angry.
Scene 4 = comedy escalates.
Scene 5 = unexpected funny punchline directly to camera.

CHARACTER:
The SAME orange cat must appear in every scene.

VISUAL STYLE:
Realistic, cinematic, Indian middle-class home.

EXACT JSON SCHEMA:

{
  "title": "string",
  "hook": "string",
  "language": "Hinglish",
  "duration_seconds": 20,
  "format": "9:16",
  "style": ["realistic", "cinematic", "funny"],
  "character": {
    "name": "string",
    "appearance": "string",
    "personality": "string"
  },
  "scenes": [
    {
      "scene_number": 1,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 2,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 3,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 4,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 5,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    }
  ]
}

IMPORTANT:
- Exactly 5 scenes.
- Every scene exactly 4 seconds.
- Total exactly 20 seconds.
- Dialogue must be Hinglish.
- Each dialogue should be short enough to speak naturally within 4 seconds.
- No English-only dialogue.
- No explanations.
- No notes.
- No extra fields.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

device = next(model.parameters()).device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

print("\n🧠 Generating improved Reel plan...")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.25,
        top_p=0.80,
        repetition_penalty=1.08
    )

raw = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("\n📄 RAW OUTPUT:")
print("-" * 70)
print(raw)
print("-" * 70)

# ------------------------------------------------------------
# JSON EXTRACTION
# ------------------------------------------------------------

match = re.search(r"\{.*\}", raw, re.DOTALL)

assert match, "No JSON object found."

plan = json.loads(match.group(0))

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

assert plan["language"] == "Hinglish"
assert plan["duration_seconds"] == 20
assert plan["format"] == "9:16"

assert len(plan["scenes"]) == 5

for i, scene in enumerate(plan["scenes"], 1):

    assert scene["scene_number"] == i
    assert scene["duration_seconds"] == 4

    assert scene["dialogue"].strip()
    assert scene["visual_prompt"].strip()
    assert scene["action"].strip()

assert sum(
    s["duration_seconds"]
    for s in plan["scenes"]
) == 20

# ------------------------------------------------------------
# SIMPLE QUALITY CHECKS
# ------------------------------------------------------------

dialogues = [
    s["dialogue"]
    for s in plan["scenes"]
]

english_markers = [
    " why ",
    " the ",
    " this ",
    " that ",
    " because ",
    " everyone ",
    " clean this place "
]

combined = " " + " ".join(dialogues).lower() + " "

english_marker_count = sum(
    combined.count(x)
    for x in english_markers
)

print("\n🔎 QUALITY CHECK")
print("Scenes:", len(plan["scenes"]))
print("Duration:", sum(
    s["duration_seconds"]
    for s in plan["scenes"]
))
print("English-marker count:", english_marker_count)

print("\n🎭 FINAL DIALOGUES:")

for scene in plan["scenes"]:
    print(
        f"Scene {scene['scene_number']}: "
        f"{scene['dialogue']}"
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUT = Path("/kaggle/working/reel_director_quality_locked.json")

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(
        plan,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n💾 Saved:", OUT)

print("\n" + "=" * 70)
print("🎉 KAGGLE-SAFE-11.1 PASS — QUALITY STRUCTURE VERIFIED")
print("=" * 70)

🎬 REEL DIRECTOR — QUALITY LOCK TEST


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


🧠 Generating improved Reel plan...

📄 RAW OUTPUT:
----------------------------------------------------------------------
- The title and format strings must match the exact ones requested. {"title": "Orange Cat's Messy Home", "hook": "Why does everyone expect me to clean?", "language": "Hinglish", "duration_seconds": 20, "format": "9:16", "style": ["realistic", "cinematic", "funny"], "character": {"name": "Orange Cat", "appearance": "Orange cat with a mop of fur", "personality": "Curious, persistent"}, "scenes": [{"scene_number": 1, "duration_seconds": 4, "visual_prompt": "Orange Cat enters the room.", "dialogue": "Why do people always expect me to clean?", "action": "The cat looks around the room."}, {"scene_number": 2, "duration_seconds": 4, "visual_prompt": "Orange Cat starts cleaning.", "dialogue": "I can't believe they think I'm supposed to clean this mess!", "action": "The cat uses its paw to sweep dust off the floor."}, {"scene_number": 3, "duration_seconds": 4, "visual_prompt"

JSONDecodeError: Expecting ',' delimiter: line 1 column 915 (char 914)

In [16]:
# ============================================================
# KAGGLE-SAFE-11.2 : ROBUST REEL DIRECTOR JSON PIPELINE
# ============================================================

import json
import re
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 70)
print("🎬 REEL DIRECTOR — ROBUST JSON PIPELINE")
print("=" * 70)

QWEN = Path(
    "/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen"
)

# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

print("\n🧠 Loading Qwen...")

tokenizer = AutoTokenizer.from_pretrained(
    str(QWEN),
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    str(QWEN),
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("✅ Qwen loaded")

# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------

prompt = r"""
You are a professional viral Instagram Reel Director.

IMPORTANT:
Return ONLY valid JSON.
The first character must be {.
The last character must be }.
Do not write anything before or after the JSON.
Do not use markdown.
Do not use comments.

Create a funny 20-second Instagram Reel.

CONCEPT:
A realistic orange cat lives in an Indian middle-class home.
The home is messy.
Everyone expects the cat to clean it.
The cat becomes increasingly frustrated.

LANGUAGE:
Natural Indian Hinglish.
Dialogue should sound casual and funny.
Mix Hindi and English naturally.
Do not make dialogue mostly English.
Do not use formal Hindi.

COMEDY:
Scene 1: strong hook.
Scene 2: comedy escalates.
Scene 3: cat becomes angry.
Scene 4: situation becomes even more ridiculous.
Scene 5: unexpected punchline directly to camera.

CHARACTER:
The SAME orange cat appears in every scene.

FORMAT:
20 seconds total.
5 scenes.
Each scene exactly 4 seconds.
9:16.
Realistic, cinematic, funny.

Return exactly this JSON structure:

{
  "title": "string",
  "hook": "string",
  "language": "Hinglish",
  "duration_seconds": 20,
  "format": "9:16",
  "style": [
    "realistic",
    "cinematic",
    "funny"
  ],
  "character": {
    "name": "string",
    "appearance": "string",
    "personality": "string"
  },
  "scenes": [
    {
      "scene_number": 1,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 2,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 3,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 4,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 5,
      "duration_seconds": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    }
  ]
}

Rules:
- Exactly 5 scenes.
- Exactly 4 seconds per scene.
- Exactly 20 seconds total.
- All fields must be strings except numeric fields.
- No extra fields.
- No newline-sensitive formatting inside strings.
- Escape all internal quotation marks correctly.
- Use short dialogue suitable for 4 seconds.
"""

# ------------------------------------------------------------
# GENERATE
# ------------------------------------------------------------

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

device = next(model.parameters()).device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

print("\n🎬 Generating...")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=False,
        repetition_penalty=1.05
    )

raw = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("\n📄 RAW MODEL OUTPUT")
print("-" * 70)
print(raw)
print("-" * 70)

# ------------------------------------------------------------
# EXTRACT JSON
# ------------------------------------------------------------

start = raw.find("{")
end = raw.rfind("}")

assert start != -1, "No opening JSON object found."
assert end != -1, "No closing JSON object found."
assert end > start, "Invalid JSON boundaries."

json_text = raw[start:end + 1]

# ------------------------------------------------------------
# FIRST JSON PARSE
# ------------------------------------------------------------

try:

    plan = json.loads(json_text)

    print("\n✅ JSON parse: PASS")

except json.JSONDecodeError as e:

    print("\n⚠️ JSON parse failed.")
    print("Error:", e)

    # --------------------------------------------------------
    # BASIC REPAIR
    # --------------------------------------------------------

    repaired = json_text

    # Remove trailing commas before } or ]
    repaired = re.sub(
        r",\s*([}\]])",
        r"\1",
        repaired
    )

    # Normalize smart quotes
    repaired = (
        repaired
        .replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )

    # Try repaired JSON
    try:

        plan = json.loads(repaired)

        print("✅ JSON repair: PASS")

    except json.JSONDecodeError as e2:

        print("\n❌ JSON repair failed.")
        print("Repair error:", e2)

        # Save raw output for debugging
        debug_path = Path(
            "/kaggle/working/reel_director_raw_failed.txt"
        )

        debug_path.write_text(
            raw,
            encoding="utf-8"
        )

        print("\n💾 Failed raw output saved:")
        print(debug_path)

        raise

# ------------------------------------------------------------
# STRUCTURE VALIDATION
# ------------------------------------------------------------

required_top = [
    "title",
    "hook",
    "language",
    "duration_seconds",
    "format",
    "style",
    "character",
    "scenes"
]

for key in required_top:
    assert key in plan, f"Missing top-level field: {key}"

assert isinstance(plan["title"], str)
assert isinstance(plan["hook"], str)

assert plan["language"] == "Hinglish"
assert plan["duration_seconds"] == 20
assert plan["format"] == "9:16"

assert isinstance(plan["style"], list)
assert isinstance(plan["character"], dict)
assert isinstance(plan["scenes"], list)

assert len(plan["scenes"]) == 5

# ------------------------------------------------------------
# SCENE VALIDATION
# ------------------------------------------------------------

scene_fields = [
    "scene_number",
    "duration_seconds",
    "visual_prompt",
    "dialogue",
    "action"
]

for i, scene in enumerate(plan["scenes"], 1):

    for field in scene_fields:
        assert field in scene, (
            f"Scene {i}: missing field '{field}'"
        )

    assert scene["scene_number"] == i

    assert scene["duration_seconds"] == 4

    assert isinstance(scene["visual_prompt"], str)
    assert isinstance(scene["dialogue"], str)
    assert isinstance(scene["action"], str)

    assert scene["visual_prompt"].strip()
    assert scene["dialogue"].strip()
    assert scene["action"].strip()

# ------------------------------------------------------------
# DURATION VALIDATION
# ------------------------------------------------------------

total_duration = sum(
    scene["duration_seconds"]
    for scene in plan["scenes"]
)

assert total_duration == 20

# ------------------------------------------------------------
# CHARACTER CONSISTENCY CHECK
# ------------------------------------------------------------

character_text = (
    plan["character"]["appearance"]
    + " "
    + " ".join(
        scene["visual_prompt"]
        for scene in plan["scenes"]
    )
).lower()

assert "orange" in character_text
assert "cat" in character_text

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUT = Path(
    "/kaggle/working/reel_director_quality_locked.json"
)

with open(
    OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        plan,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# DISPLAY RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 KAGGLE-SAFE-11.2 PASS")
print("=" * 70)

print("\n🎬 TITLE:")
print(plan["title"])

print("\n🪝 HOOK:")
print(plan["hook"])

print("\n🎭 CHARACTER:")
print(plan["character"])

print("\n🎞️ SCENES:")

for scene in plan["scenes"]:

    print(
        f"\nScene {scene['scene_number']} "
        f"({scene['duration_seconds']} sec)"
    )

    print("Dialogue:", scene["dialogue"])
    print("Action:", scene["action"])

print("\n💾 Saved:")
print(OUT)

print("\n" + "=" * 70)

🎬 REEL DIRECTOR — ROBUST JSON PIPELINE

🧠 Loading Qwen...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Qwen loaded

🎬 Generating...

📄 RAW MODEL OUTPUT
----------------------------------------------------------------------
- The humor should come from the cat's frustration and the absurdity of the situation. Here's the JSON structure for the 20-second Instagram Reel with the specified concept, language, style, and scenes:

```json
{
  "title": "Orange Cat in a Messy Home",
  "hook": "Orange cat in a messy house, everyone expects him to clean it.",
  "language": "Hinglish",
  "duration_seconds": "20",
  "format": "9:16",
  "style": ["realistic", "cinematic", "funny"],
  "character": {
    "name": "Orange Cat",
    "appearance": "Orange cat with a fluffy tail and expressive eyes",
    "personality": "Fluffy, cute, and slightly frustrated"
  },
  "scenes": [
    {
      "scene_number": 1,
      "duration_seconds": "4",
      "visual_prompt": "Orange cat lounges on a dirty sofa, looking at a mess.",
      "dialogue": "Orange Cat: 'Why do I have to clean this? It's my home too!'",
      "a

AssertionError: 

In [17]:
# ============================================================
# KAGGLE-SAFE-11.3
# CONTROLLER-LOCKED REEL DIRECTOR
# ============================================================

import json
import re
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 70)
print("🎬 PERSONAL AI — CONTROLLER-LOCKED REEL DIRECTOR")
print("=" * 70)

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

QWEN = Path(
    "/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen"
)

assert QWEN.exists(), "Qwen dataset not found."

# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

print("\n🧠 Loading Qwen...")

tokenizer = AutoTokenizer.from_pretrained(
    str(QWEN),
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    str(QWEN),
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("✅ Qwen loaded")

# ------------------------------------------------------------
# CONTROLLER CONSTANTS
# ------------------------------------------------------------

TARGET_DURATION = 20
TARGET_FORMAT = "9:16"
SCENE_COUNT = 5
SCENE_DURATION = 4

# ------------------------------------------------------------
# CREATIVE PROMPT
# ------------------------------------------------------------

prompt = r"""
You are the creative brain inside a Reel production system.

Return ONLY one valid JSON object.
No markdown.
No explanation.
No text before or after JSON.

Your job is ONLY to generate creative content.
The production controller will automatically enforce:
- 20 second duration
- 9:16 format
- exactly 5 scenes
- exactly 4 seconds per scene

Therefore do NOT decide or modify duration or format.

Create a funny viral Instagram Reel.

CONCEPT:

A realistic orange cat lives in an Indian middle-class home.
The house is extremely messy.
Everyone expects the cat to clean the house.
The cat becomes increasingly frustrated.

LANGUAGE:

Natural Indian Hinglish.
The dialogue should sound like casual Indian speech.
Use Hindi and English naturally.
Avoid formal Hindi.
Avoid English-only dialogue.

COMEDY ARC:

Scene 1:
Strong curiosity hook.

Scene 2:
The situation becomes more ridiculous.

Scene 3:
The cat becomes genuinely frustrated.

Scene 4:
Comedy escalates.

Scene 5:
Unexpected funny punchline directly toward the camera.

CHARACTER:

The SAME orange cat throughout the entire Reel.

VISUAL STYLE:

Realistic.
Cinematic.
Indian middle-class home.
Funny expressions.
Natural physical comedy.

RETURN EXACTLY THIS CREATIVE JSON:

{
  "title": "string",
  "hook": "string",
  "character": {
    "name": "string",
    "appearance": "string",
    "personality": "string"
  },
  "scenes": [
    {
      "scene_number": 1,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 2,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 3,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 5,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    }
  ]
}

IMPORTANT:

- Exactly 5 scenes.
- scene_number must be 1 through 5.
- Keep dialogue short enough for approximately 4 seconds.
- Keep the orange cat visually consistent.
- Final scene must contain a punchline.
- No extra fields.
"""

# ------------------------------------------------------------
# GENERATION
# ------------------------------------------------------------

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

device = next(model.parameters()).device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

print("\n🎬 Generating creative Reel plan...")

with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=False,
        repetition_penalty=1.05
    )

raw = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("\n📄 RAW MODEL OUTPUT")
print("-" * 70)
print(raw)
print("-" * 70)

# ------------------------------------------------------------
# EXTRACT JSON
# ------------------------------------------------------------

start = raw.find("{")
end = raw.rfind("}")

assert start >= 0, "No JSON object found."
assert end > start, "Invalid JSON boundaries."

json_text = raw[start:end + 1]

try:

    creative = json.loads(json_text)

except json.JSONDecodeError as e:

    print("\n⚠️ Initial JSON parse failed.")
    print("Error:", e)

    # Basic safe repair
    repaired = re.sub(
        r",\s*([}\]])",
        r"\1",
        json_text
    )

    repaired = (
        repaired
        .replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )

    creative = json.loads(repaired)

print("\n✅ Creative JSON parsed")

# ------------------------------------------------------------
# CREATIVE VALIDATION
# ------------------------------------------------------------

assert isinstance(creative, dict)

assert "title" in creative
assert "hook" in creative
assert "character" in creative
assert "scenes" in creative

assert isinstance(creative["title"], str)
assert isinstance(creative["hook"], str)
assert isinstance(creative["character"], dict)
assert isinstance(creative["scenes"], list)

assert len(creative["scenes"]) == SCENE_COUNT

for i, scene in enumerate(
    creative["scenes"],
    start=1
):

    assert scene["scene_number"] == i

    assert scene["visual_prompt"].strip()
    assert scene["dialogue"].strip()
    assert scene["action"].strip()

# ------------------------------------------------------------
# PRODUCTION CONTROLLER
# ------------------------------------------------------------

print("\n⚙️ Applying production controller...")

final_plan = {
    "title": creative["title"],
    "hook": creative["hook"],
    "language": "Hinglish",
    "duration_seconds": TARGET_DURATION,
    "format": TARGET_FORMAT,
    "style": [
        "realistic",
        "cinematic",
        "funny"
    ],
    "character": creative["character"],
    "scenes": []
}

for i, scene in enumerate(
    creative["scenes"],
    start=1
):

    final_plan["scenes"].append({
        "scene_number": i,
        "duration_seconds": SCENE_DURATION,
        "visual_prompt": scene["visual_prompt"],
        "dialogue": scene["dialogue"],
        "action": scene["action"]
    })

# ------------------------------------------------------------
# FINAL HARD VALIDATION
# ------------------------------------------------------------

assert final_plan["duration_seconds"] == 20
assert final_plan["format"] == "9:16"
assert final_plan["language"] == "Hinglish"

assert len(final_plan["scenes"]) == 5

assert sum(
    s["duration_seconds"]
    for s in final_plan["scenes"]
) == 20

for i, scene in enumerate(
    final_plan["scenes"],
    start=1
):

    assert scene["scene_number"] == i
    assert scene["duration_seconds"] == 4

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUT = Path(
    "/kaggle/working/reel_director_final_plan.json"
)

with open(
    OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_plan,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 KAGGLE-SAFE-11.3 PASS")
print("=" * 70)

print("\n🎬 TITLE:")
print(final_plan["title"])

print("\n🪝 HOOK:")
print(final_plan["hook"])

print("\n⏱️ Duration:")
print(final_plan["duration_seconds"], "seconds")

print("\n📱 Format:")
print(final_plan["format"])

print("\n🎭 CHARACTER:")
print(final_plan["character"])

print("\n🎞️ SCENES:")

for scene in final_plan["scenes"]:

    print(
        f"\nScene {scene['scene_number']} "
        f"({scene['duration_seconds']} sec)"
    )

    print("Dialogue:", scene["dialogue"])
    print("Action:", scene["action"])

print("\n💾 Saved:")
print(OUT)

print("\n" + "=" * 70)

🎬 PERSONAL AI — CONTROLLER-LOCKED REEL DIRECTOR

🧠 Loading Qwen...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Qwen loaded

🎬 Generating creative Reel plan...

📄 RAW MODEL OUTPUT
----------------------------------------------------------------------
- The title, hook, character details, and scenes must be filled out. {
  "title": "Orange Cat's Messy Home",
  "hook": "Why does everyone expect me to clean?",
  "character": {
    "name": "Orange Cat",
    "appearance": "Orange cat with a curious expression, wearing a t-shirt and jeans.",
    "personality": "Curious, mischievous, and slightly frustrated."
  },
  "scenes": [
    {
      "scene_number": 1,
      "visual_prompt": "The cat is sitting on the couch, looking at the mess around it.",
      "dialogue": "Why does everyone expect me to clean?",
      "action": "The cat looks up at the camera, its eyes wide with curiosity."
    },
    {
      "scene_number": 2,
      "visual_prompt": "The cat starts to pick up toys, but drops them everywhere.",
      "dialogue": "I can't believe they expect me to clean!",
      "action": "The cat picks up a 

In [18]:
# ============================================================
# KAGGLE-SAFE-11.4
# VIRAL HINGLISH REEL CREATIVE DIRECTOR
# ============================================================

import json
import re
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 70)
print("🎬 PERSONAL AI — VIRAL HINGLISH CREATIVE DIRECTOR")
print("=" * 70)

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

QWEN = Path("/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen")
assert QWEN.exists(), "Qwen dataset not found."

# ------------------------------------------------------------
# LOAD QWEN
# ------------------------------------------------------------

print("\n🧠 Loading Qwen...")

tokenizer = AutoTokenizer.from_pretrained(
    str(QWEN),
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    str(QWEN),
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("✅ Qwen loaded")

# ------------------------------------------------------------
# HARD PRODUCTION CONSTANTS
# ------------------------------------------------------------

DURATION = 20
FORMAT = "9:16"
SCENE_COUNT = 5
SCENE_DURATION = 4
LANGUAGE = "Hinglish"

# ------------------------------------------------------------
# CREATIVE DIRECTOR PROMPT
# ------------------------------------------------------------

prompt = r"""
You are the VIRAL CREATIVE DIRECTOR of a professional Indian Instagram
Reels production system.

Your job is to create a FUNNY, HIGH-RETENTION, NATURAL HINGLISH Reel.

IMPORTANT:
Return ONLY the JSON object.
Do NOT write explanations.
Do NOT write markdown.
Do NOT write bullet points.
Do NOT write anything before or after the JSON.

==================================================
REEL CONCEPT
==================================================

A realistic orange cat lives in a messy Indian middle-class home.

The family keeps expecting the cat to clean the house.

The cat becomes increasingly irritated.

The comedy should feel like an Indian person complaining about a
normal household problem, except the character is a talking cat.

==================================================
CHARACTER
==================================================

Same character throughout:

Realistic orange domestic cat.

Personality:
- sarcastic
- dramatic
- mischievous
- easily irritated
- unexpectedly confident

The cat behaves naturally like a real cat but communicates like
a funny Indian human.

==================================================
LANGUAGE
==================================================

Use NATURAL HINGLISH.

The dialogue must contain a natural mixture of Hindi and English.

Avoid:
- formal Hindi
- pure English
- robotic dialogue
- textbook Hindi
- unnatural translations

The dialogue should sound like something an Indian person might
actually say casually.

==================================================
COMEDY STRUCTURE
==================================================

SCENE 1 — HOOK

Immediately create curiosity.

The viewer should think:
"What is this cat complaining about?"

SCENE 2 — SETUP

Reveal the ridiculous problem.

SCENE 3 — ESCALATION

The cat becomes more angry and dramatic.

SCENE 4 — PEAK COMEDY

The situation becomes absurd.

SCENE 5 — PUNCHLINE

The cat looks directly toward the camera and delivers an
unexpected punchline.

The final line must feel like a joke, not merely the ending
of the story.

==================================================
DIALOGUE RULES
==================================================

Each scene has ONE short spoken line.

Each dialogue should fit approximately 4 seconds.

Keep dialogue conversational.

Use expressions such as:
"yaar", "bhai", "seriously", "bas", "kya yaar", etc.
ONLY where naturally appropriate.

Do not force slang into every sentence.

==================================================
VISUAL RULES
==================================================

Every scene must describe a visually interesting action.

Do not simply write:
"cat is standing."

Instead show actions such as:
- throwing a toy
- dragging a cloth
- staring at a messy room
- knocking something over
- dramatically dropping a mop
- looking directly at camera
- reacting with exaggerated facial expression

The SAME orange cat must remain visually consistent.

==================================================
PUNCHLINE RULE
==================================================

Scene 5 must contain the strongest joke.

The punchline should have a surprise.

Example direction:

The cat refuses to work unless it gets paid.

But DO NOT copy this exact example.

Create your own punchline.

==================================================
OUTPUT SCHEMA
==================================================

{
  "title": "short catchy title",
  "hook": "short hook",
  "character": {
    "name": "short character name",
    "appearance": "consistent visual description",
    "personality": "character personality"
  },
  "scenes": [
    {
      "scene_number": 1,
      "visual_prompt": "cinematic visual description",
      "dialogue": "natural Hinglish dialogue",
      "action": "specific physical action"
    },
    {
      "scene_number": 2,
      "visual_prompt": "cinematic visual description",
      "dialogue": "natural Hinglish dialogue",
      "action": "specific physical action"
    },
    {
      "scene_number": 3,
      "visual_prompt": "cinematic visual description",
      "dialogue": "natural Hinglish dialogue",
      "action": "specific physical action"
    },
    {
      "scene_number": 4,
      "visual_prompt": "cinematic visual description",
      "dialogue": "natural Hinglish dialogue",
      "action": "specific physical action"
    },
    {
      "scene_number": 5,
      "visual_prompt": "cinematic visual description with direct-to-camera ending",
      "dialogue": "strong unexpected Hinglish punchline",
      "action": "specific final comedic action"
    }
  ]
}

==================================================
FINAL REQUIREMENTS
==================================================

Exactly 5 scenes.

Scene numbers must be 1,2,3,4,5.

No empty fields.

Dialogue must be Hinglish.

Scene 5 must contain the punchline.

Maintain the SAME orange cat.

Do not include duration fields.

Do not include format fields.

Do not include extra JSON fields.
"""

# ------------------------------------------------------------
# GENERATE
# ------------------------------------------------------------

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

device = next(model.parameters()).device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

print("\n🎨 Generating viral Hinglish concept...")

with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=False,
        repetition_penalty=1.08
    )

raw = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print("\n📄 RAW OUTPUT")
print("-" * 70)
print(raw)
print("-" * 70)

# ------------------------------------------------------------
# EXTRACT JSON
# ------------------------------------------------------------

start = raw.find("{")
end = raw.rfind("}")

assert start != -1, "No JSON object found."
assert end > start, "Invalid JSON boundaries."

json_text = raw[start:end + 1]

# ------------------------------------------------------------
# PARSE
# ------------------------------------------------------------

try:

    creative = json.loads(json_text)

except json.JSONDecodeError:

    print("\n⚠️ Repairing malformed JSON...")

    repaired = re.sub(
        r",\s*([}\]])",
        r"\1",
        json_text
    )

    repaired = (
        repaired
        .replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )

    creative = json.loads(repaired)

print("\n✅ JSON parsed")

# ------------------------------------------------------------
# STRUCTURAL VALIDATION
# ------------------------------------------------------------

assert isinstance(creative, dict)

for key in [
    "title",
    "hook",
    "character",
    "scenes"
]:
    assert key in creative, f"Missing field: {key}"

assert isinstance(creative["character"], dict)
assert isinstance(creative["scenes"], list)

assert len(creative["scenes"]) == SCENE_COUNT

required_character = [
    "name",
    "appearance",
    "personality"
]

for key in required_character:
    assert key in creative["character"]
    assert str(creative["character"][key]).strip()

for i, scene in enumerate(
    creative["scenes"],
    start=1
):

    assert scene["scene_number"] == i

    assert str(
        scene["visual_prompt"]
    ).strip()

    assert str(
        scene["dialogue"]
    ).strip()

    assert str(
        scene["action"]
    ).strip()

# ------------------------------------------------------------
# CONTROLLER LOCK
# ------------------------------------------------------------

final_plan = {
    "title": creative["title"],
    "hook": creative["hook"],
    "language": LANGUAGE,
    "duration_seconds": DURATION,
    "format": FORMAT,
    "style": [
        "realistic",
        "cinematic",
        "comedy"
    ],
    "character": creative["character"],
    "scenes": []
}

for scene in creative["scenes"]:

    final_plan["scenes"].append({
        "scene_number": scene["scene_number"],
        "duration_seconds": SCENE_DURATION,
        "visual_prompt": scene["visual_prompt"],
        "dialogue": scene["dialogue"],
        "action": scene["action"]
    })

# ------------------------------------------------------------
# HARD PRODUCTION VALIDATION
# ------------------------------------------------------------

assert final_plan["language"] == "Hinglish"
assert final_plan["duration_seconds"] == 20
assert final_plan["format"] == "9:16"

assert len(final_plan["scenes"]) == 5

assert sum(
    scene["duration_seconds"]
    for scene in final_plan["scenes"]
) == 20

for i, scene in enumerate(
    final_plan["scenes"],
    start=1
):

    assert scene["scene_number"] == i
    assert scene["duration_seconds"] == 4

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUT = Path(
    "/kaggle/working/"
    "reel_director_viral_hinglish_plan.json"
)

with open(
    OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_plan,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 KAGGLE-SAFE-11.4 PASS")
print("=" * 70)

print("\n🎬 TITLE:")
print(final_plan["title"])

print("\n🪝 HOOK:")
print(final_plan["hook"])

print("\n🗣️ LANGUAGE:")
print(final_plan["language"])

print("\n⏱️ DURATION:")
print(final_plan["duration_seconds"], "seconds")

print("\n📱 FORMAT:")
print(final_plan["format"])

print("\n🎭 CHARACTER:")
print(final_plan["character"]["name"])
print(final_plan["character"]["appearance"])

print("\n🎞️ REEL:")
print("-" * 70)

for scene in final_plan["scenes"]:

    print(
        f"\nScene {scene['scene_number']} "
        f"({scene['duration_seconds']} sec)"
    )

    print("Dialogue :", scene["dialogue"])
    print("Action   :", scene["action"])
    print("Visual   :", scene["visual_prompt"])

print("\n" + "-" * 70)
print("💾 SAVED:")
print(OUT)

print("\n" + "=" * 70)

🎬 PERSONAL AI — VIRAL HINGLISH CREATIVE DIRECTOR

🧠 Loading Qwen...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Qwen loaded

🎨 Generating viral Hinglish concept...

📄 RAW OUTPUT
----------------------------------------------------------------------
```json
{
  "title": "Cat's Complaint",
  "hook": "What's this cat saying?",
  "character": {
    "name": "Orange Cat",
    "appearance": "orange cat with expressive face",
    "personality": "sarcastic, dramatic, mischievous, easily irritated, unexpectedly confident"
  },
  "scenes": [
    {
      "scene_number": 1,
      "visual_prompt": "Orange cat stands on a cluttered floor, looking annoyed.",
      "dialogue": "Orange Cat: 'Yaar, bhai, seriously, why do I have to clean this mess? It’s already dirty! And you expect me to do it all day long?!'",
      "action": "Orange Cat throws a small toy across the room."
    },
    {
      "scene_number": 2,
      "visual_prompt": "Orange Cat sits on a chair, fur matted, looking frustrated.",
      "dialogue": "Orange Cat: 'Bhai, seriously, how can I clean when there’s no time for even a little bit of effor

In [19]:
# ============================================================
# KAGGLE-SAFE-11.5
# REEL QUALITY JUDGE + AUTO-REPAIR
# ============================================================

import json
import re
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

print("=" * 70)
print("🎯 PERSONAL AI — REEL QUALITY JUDGE + AUTO-REPAIR")
print("=" * 70)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

QWEN = Path(
    "/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen"
)

INPUT_PLAN = Path(
    "/kaggle/working/reel_director_viral_hinglish_plan.json"
)

OUTPUT_PLAN = Path(
    "/kaggle/working/reel_director_quality_locked.json"
)

assert QWEN.exists(), "Qwen dataset not found."
assert INPUT_PLAN.exists(), "11.4 Reel plan not found."

# ------------------------------------------------------------
# LOAD EXISTING PLAN
# ------------------------------------------------------------

with open(INPUT_PLAN, "r", encoding="utf-8") as f:
    original_plan = json.load(f)

print("\n📂 Existing Reel plan loaded")

# ------------------------------------------------------------
# LOAD QWEN
# ------------------------------------------------------------

print("\n🧠 Loading Qwen...")

tokenizer = AutoTokenizer.from_pretrained(
    str(QWEN),
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    str(QWEN),
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

print("✅ Qwen loaded")

# ------------------------------------------------------------
# QUALITY JUDGE
# ------------------------------------------------------------

scenes = original_plan["scenes"]

issues = []

# Basic structure
if len(scenes) != 5:
    issues.append("Exactly 5 scenes required.")

# Duration
if original_plan.get("duration_seconds") != 20:
    issues.append("Total duration must be 20 seconds.")

if original_plan.get("format") != "9:16":
    issues.append("Format must be 9:16.")

# Scene durations
for scene in scenes:

    if scene.get("duration_seconds") != 4:
        issues.append(
            f"Scene {scene.get('scene_number')} "
            "must be exactly 4 seconds."
        )

# Dialogue quality
for scene in scenes:

    dialogue = str(scene.get("dialogue", "")).strip()

    # Remove speaker prefix for word count
    clean_dialogue = re.sub(
        r"^[^:]+:\s*",
        "",
        dialogue
    )

    words = clean_dialogue.split()

    if len(words) > 12:
        issues.append(
            f"Scene {scene.get('scene_number')} dialogue "
            f"is too long ({len(words)} words)."
        )

# Repeated phrases
dialogues = [
    str(s.get("dialogue", "")).lower()
    for s in scenes
]

joined = " ".join(dialogues)

for phrase in [
    "bhai, seriously",
    "seriously",
    "i can't believe",
    "this is so frustrating"
]:

    count = joined.count(phrase)

    if count >= 2:
        issues.append(
            f"Repeated phrase detected: '{phrase}'"
        )

# Scene 4/5 similarity
if len(scenes) >= 5:

    d4 = scenes[3]["dialogue"].lower()
    d5 = scenes[4]["dialogue"].lower()

    common_words = set(d4.split()) & set(d5.split())

    if len(common_words) >= 6:
        issues.append(
            "Scene 4 and Scene 5 are too similar."
        )

# Punchline heuristic
final_dialogue = str(
    scenes[-1].get("dialogue", "")
).lower()

punchline_words = [
    "salary",
    "paid",
    "payment",
    "money",
    "maid",
    "strike",
    "job",
    "responsibility",
    "boss",
    "office"
]

if not any(
    word in final_dialogue
    for word in punchline_words
):
    issues.append(
        "Final scene does not contain a sufficiently "
        "clear surprise/comedy punchline."
    )

# ------------------------------------------------------------
# SHOW JUDGEMENT
# ------------------------------------------------------------

print("\n🔍 QUALITY AUDIT")
print("-" * 70)

if issues:

    print(f"⚠️ {len(issues)} quality issue(s) detected:")

    for i, issue in enumerate(issues, 1):
        print(f"{i}. {issue}")

else:

    print("✅ No major quality issues detected.")

# ------------------------------------------------------------
# AUTO-REPAIR IF REQUIRED
# ------------------------------------------------------------

if issues:

    print("\n🔧 AUTO-REPAIR ACTIVATED")

    repair_prompt = f"""
You are a professional Indian Instagram Reel script doctor.

Rewrite the following Reel into a MUCH BETTER version.

Return ONLY valid JSON.
No markdown.
No explanation.
No text before JSON.
No text after JSON.

==================================================
REEL GOAL
==================================================

Create a funny, highly watchable 20-second Reel.

Character:
A realistic orange domestic cat living in an Indian
middle-class home.

The cat talks like a funny Indian person.

==================================================
STRICT RULES
==================================================

1. Exactly 5 scenes.
2. Exactly 4 seconds per scene.
3. Total duration = 20 seconds.
4. Language = natural Indian Hinglish.
5. Each dialogue must be SHORT.
6. Maximum 10 spoken words per scene.
7. Never repeat "bhai, seriously".
8. Never repeat the same joke.
9. Every scene must advance the story.
10. Scene 1 must hook immediately.
11. Scene 2 establishes the problem.
12. Scene 3 escalates the problem.
13. Scene 4 reaches peak comedy.
14. Scene 5 must contain a SURPRISE punchline.
15. Scene 5 must look directly toward camera.
16. Keep the same orange cat in every scene.
17. Visual actions must be physically understandable.
18. Avoid complicated impossible actions.
19. Dialogue must sound conversational, not translated.
20. Do not use speaker labels such as "Orange Cat:".
21. Do not include duration or format inside scene objects.

==================================================
COMEDY STYLE
==================================================

Indian household comedy.

The cat is frustrated because everyone expects it to
clean the house.

The joke should feel relatable to Indian viewers.

The final punchline should be unexpected.

Do NOT copy this example:

"Main billi hoon bhai, maid nahi."

Create an original punchline.

==================================================
CURRENT REEL
==================================================

{json.dumps(original_plan, ensure_ascii=False, indent=2)}

==================================================
OUTPUT
==================================================

{{
  "title": "short catchy title",
  "hook": "short hook",
  "character": {{
    "name": "Orange Cat",
    "appearance": "consistent realistic orange domestic cat",
    "personality": "sarcastic, dramatic, mischievous"
  }},
  "scenes": [
    {{
      "scene_number": 1,
      "visual_prompt": "short cinematic visual",
      "dialogue": "short natural Hinglish line",
      "action": "clear physical action"
    }},
    {{
      "scene_number": 2,
      "visual_prompt": "short cinematic visual",
      "dialogue": "short natural Hinglish line",
      "action": "clear physical action"
    }},
    {{
      "scene_number": 3,
      "visual_prompt": "short cinematic visual",
      "dialogue": "short natural Hinglish line",
      "action": "clear physical action"
    }},
    {{
      "scene_number": 4,
      "visual_prompt": "short cinematic visual",
      "dialogue": "short natural Hinglish line",
      "action": "clear physical action"
    }},
    {{
      "scene_number": 5,
      "visual_prompt": "direct-to-camera cinematic ending",
      "dialogue": "short unexpected Hinglish punchline",
      "action": "clear comedic final action"
    }}
  ]
}}
"""

    inputs = tokenizer(
        repair_prompt,
        return_tensors="pt"
    )

    device = next(model.parameters()).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    print("\n🎨 Regenerating improved Reel...")

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=1100,
            do_sample=False,
            repetition_penalty=1.08
        )

    raw = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("\n📄 REPAIRED RAW OUTPUT")
    print("-" * 70)
    print(raw)
    print("-" * 70)

    # --------------------------------------------------------
    # EXTRACT JSON
    # --------------------------------------------------------

    start = raw.find("{")
    end = raw.rfind("}")

    assert start >= 0, "No JSON found after repair."
    assert end > start, "Invalid repaired JSON."

    json_text = raw[start:end + 1]

    try:

        repaired_plan = json.loads(json_text)

    except json.JSONDecodeError:

        repaired_text = re.sub(
            r",\s*([}\]])",
            r"\1",
            json_text
        )

        repaired_text = (
            repaired_text
            .replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
        )

        repaired_plan = json.loads(repaired_text)

else:

    repaired_plan = original_plan

# ------------------------------------------------------------
# CONTROLLER LOCK
# ------------------------------------------------------------

final_plan = {
    "title": repaired_plan["title"],
    "hook": repaired_plan["hook"],
    "language": "Hinglish",
    "duration_seconds": 20,
    "format": "9:16",
    "style": [
        "realistic",
        "cinematic",
        "comedy"
    ],
    "character": repaired_plan["character"],
    "scenes": []
}

for i, scene in enumerate(
    repaired_plan["scenes"],
    start=1
):

    dialogue = str(
        scene["dialogue"]
    ).strip()

    # Remove accidental speaker labels
    dialogue = re.sub(
        r"^[^:]{1,30}:\s*",
        "",
        dialogue
    ).strip()

    final_plan["scenes"].append({
        "scene_number": i,
        "duration_seconds": 4,
        "visual_prompt": str(
            scene["visual_prompt"]
        ).strip(),
        "dialogue": dialogue,
        "action": str(
            scene["action"]
        ).strip()
    })

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

assert final_plan["language"] == "Hinglish"
assert final_plan["duration_seconds"] == 20
assert final_plan["format"] == "9:16"

assert len(final_plan["scenes"]) == 5

assert sum(
    s["duration_seconds"]
    for s in final_plan["scenes"]
) == 20

for i, scene in enumerate(
    final_plan["scenes"],
    start=1
):

    assert scene["scene_number"] == i
    assert scene["duration_seconds"] == 4
    assert scene["dialogue"]
    assert scene["visual_prompt"]
    assert scene["action"]

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

with open(
    OUTPUT_PLAN,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_plan,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 KAGGLE-SAFE-11.5 PASS")
print("=" * 70)

print("\n🎬 TITLE:")
print(final_plan["title"])

print("\n🪝 HOOK:")
print(final_plan["hook"])

print("\n🎭 CHARACTER:")
print(final_plan["character"])

print("\n🎞️ FINAL REEL")
print("-" * 70)

for scene in final_plan["scenes"]:

    print(
        f"\nScene {scene['scene_number']} "
        f"({scene['duration_seconds']} sec)"
    )

    print("🗣️", scene["dialogue"])
    print("🎬", scene["action"])
    print("👁️", scene["visual_prompt"])

print("\n" + "-" * 70)

print("⏱️ Total:", final_plan["duration_seconds"], "seconds")
print("📱 Format:", final_plan["format"])
print("🗣️ Language:", final_plan["language"])
print("🎞️ Scenes:", len(final_plan["scenes"]))

print("\n💾 SAVED:")
print(OUTPUT_PLAN)

print("\n" + "=" * 70)

🎯 PERSONAL AI — REEL QUALITY JUDGE + AUTO-REPAIR

📂 Existing Reel plan loaded

🧠 Loading Qwen...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Qwen loaded

🔍 QUALITY AUDIT
----------------------------------------------------------------------
⚠️ 8 quality issue(s) detected:
1. Scene 1 dialogue is too long (24 words).
2. Scene 2 dialogue is too long (22 words).
3. Scene 3 dialogue is too long (21 words).
4. Scene 4 dialogue is too long (17 words).
5. Scene 5 dialogue is too long (35 words).
6. Repeated phrase detected: 'bhai, seriously'
7. Repeated phrase detected: 'seriously'
8. Scene 4 and Scene 5 are too similar.

🔧 AUTO-REPAIR ACTIVATED

🎨 Regenerating improved Reel...

📄 REPAIRED RAW OUTPUT
----------------------------------------------------------------------

Please provide the output in JSON format. The JSON should follow the structure provided above. No additional information is needed beyond the JSON object itself. The JSON should be exactly as shown without any changes or additions. The JSON should be formatted correctly according to the given rules. The JSON should not contain any spaces or newlines. The JSON sho

AssertionError: No JSON found after repair.

In [20]:
# ============================================================
# KAGGLE-SAFE-11.5.1
# PERSONAL AI — ROBUST REEL QUALITY JUDGE + AUTO-REPAIR
# ============================================================

import os, json, re, textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM

print("🎯 PERSONAL AI — ROBUST REEL QUALITY JUDGE + AUTO-REPAIR")

MODEL_DIR = "/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen"
INPUT_PLAN = "/kaggle/working/reel_director_viral_hinglish_plan.json"
OUTPUT_PLAN = "/kaggle/working/reel_director_quality_locked.json"

assert os.path.exists(INPUT_PLAN), f"Input plan missing: {INPUT_PLAN}"

# ------------------------------------------------------------
# 1. LOAD MODEL
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

print("✅ Qwen loaded")

# ------------------------------------------------------------
# 2. LOAD ORIGINAL PLAN
# ------------------------------------------------------------

with open(INPUT_PLAN, "r", encoding="utf-8") as f:
    original = json.load(f)

# ------------------------------------------------------------
# 3. QUALITY AUDIT
# ------------------------------------------------------------

issues = []

scenes = original.get("scenes", [])

if len(scenes) != 5:
    issues.append("Scene count must be exactly 5.")

if original.get("duration_sec") != 20:
    issues.append("Total duration must be exactly 20 seconds.")

if original.get("format") != "9:16":
    issues.append("Format must be exactly 9:16.")

for i, scene in enumerate(scenes, 1):

    if scene.get("duration_sec") != 4:
        issues.append(f"Scene {i} duration must be exactly 4 seconds.")

    dialogue = str(scene.get("dialogue", "")).strip()

    # Remove speaker prefixes for judging
    clean_dialogue = re.sub(
        r"^[A-Za-z ]+:\s*",
        "",
        dialogue
    ).strip()

    words = clean_dialogue.split()

    if len(words) > 10:
        issues.append(
            f"Scene {i} dialogue too long: {len(words)} words."
        )

# ------------------------------------------------------------
# Repetition detection
# ------------------------------------------------------------

all_dialogue = " ".join(
    str(s.get("dialogue", "")).lower()
    for s in scenes
)

bad_repetitions = [
    "bhai, seriously",
    "seriously",
    "i can't believe",
    "this is so frustrating",
    "you’re asking too much",
    "you're asking too much"
]

for phrase in bad_repetitions:
    if all_dialogue.count(phrase) > 1:
        issues.append(
            f"Repeated phrase detected: '{phrase}'"
        )

# ------------------------------------------------------------
# Scene 4 / 5 similarity
# ------------------------------------------------------------

if len(scenes) >= 5:

    d4 = str(scenes[3].get("dialogue", "")).lower()
    d5 = str(scenes[4].get("dialogue", "")).lower()

    words4 = set(re.findall(r"\b\w+\b", d4))
    words5 = set(re.findall(r"\b\w+\b", d5))

    if words4 and words5:
        similarity = len(words4 & words5) / max(
            1,
            len(words4 | words5)
        )

        if similarity > 0.45:
            issues.append(
                "Scene 4 and Scene 5 are too similar."
            )

# ------------------------------------------------------------
# 4. DECIDE WHETHER REPAIR IS REQUIRED
# ------------------------------------------------------------

print("\n🔍 QUALITY AUDIT")

if issues:
    print("⚠️ Issues detected:")
    for x in issues:
        print(" -", x)
else:
    print("✅ Original plan already passes basic checks.")

# ------------------------------------------------------------
# 5. STRICT REPAIR PROMPT
# ------------------------------------------------------------

repair_prompt = """
You are a professional viral Instagram Reels comedy director.

REPAIR the supplied reel plan.

IMPORTANT:
Return ONLY valid JSON.
Do NOT write explanations.
Do NOT write markdown.
Do NOT write ```json.
Do NOT write Character:
Do NOT write any text before or after JSON.

CREATIVE REQUIREMENTS:

- Indian middle-class home
- one consistent orange cat
- funny human-like talking cat
- natural Indian Hinglish
- relatable Indian humor
- short spoken dialogue
- maximum 10 spoken words per scene
- exactly 5 scenes
- exactly 4 seconds per scene
- total exactly 20 seconds
- 9:16 vertical
- no repeated "bhai, seriously"
- no repetitive dialogue
- no speaker labels
- every scene must advance the joke
- scene 1 = strong hook
- scene 2 = setup
- scene 3 = escalation
- scene 4 = comedy peak
- scene 5 = unexpected punchline
- scene 5 must NOT simply repeat scene 4
- actions must be easy for AI video generation
- visual prompts must be realistic and physically plausible

The cat should feel like ONE continuous character.

Use this exact JSON schema:

{
  "title": "string",
  "hook": "string",
  "language": "Hinglish",
  "duration_sec": 20,
  "format": "9:16",
  "character": {
    "name": "Orange Cat",
    "appearance": "string",
    "personality": "string"
  },
  "scenes": [
    {
      "scene_number": 1,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 2,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 3,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 4,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    },
    {
      "scene_number": 5,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "string",
      "action": "string"
    }
  ]
}

ORIGINAL PLAN:
""" + json.dumps(original, ensure_ascii=False, indent=2)

# ------------------------------------------------------------
# 6. ROBUST JSON EXTRACTION
# ------------------------------------------------------------

def extract_json(text):

    if not text:
        return None

    text = text.strip()

    # Remove markdown fences
    text = re.sub(r"```(?:json)?", "", text, flags=re.I)
    text = text.replace("```", "").strip()

    # First attempt: direct JSON
    try:
        return json.loads(text)
    except:
        pass

    # Locate outer JSON object
    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:

        candidate = text[start:end + 1]

        try:
            return json.loads(candidate)
        except:
            pass

        # Remove trailing commas
        candidate2 = re.sub(
            r",\s*([}\]])",
            r"\1",
            candidate
        )

        try:
            return json.loads(candidate2)
        except:
            pass

        # Normalize smart quotes
        candidate3 = (
            candidate2
            .replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
        )

        try:
            return json.loads(candidate3)
        except:
            pass

    return None


# ------------------------------------------------------------
# 7. MODEL GENERATION FUNCTION
# ------------------------------------------------------------

def generate_repair(prompt, max_new_tokens=1400):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a JSON-only creative director. "
                "Output valid JSON and nothing else."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    # Move inputs to first model device
    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None
    )

    generated = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


# ------------------------------------------------------------
# 8. FIRST REPAIR ATTEMPT
# ------------------------------------------------------------

print("\n🧠 Running first repair...")

raw = generate_repair(repair_prompt)

print("\nRAW REPAIR OUTPUT:")
print(raw[:4000])

plan = extract_json(raw)

# ------------------------------------------------------------
# 9. SECOND FALLBACK REPAIR
# ------------------------------------------------------------

if plan is None:

    print("\n⚠️ First repair did not return valid JSON.")
    print("🔁 Running emergency JSON-only repair...")

    emergency_prompt = """
Return ONLY valid JSON.

Convert the following reel concept into this exact schema.

No explanation.
No markdown.
No text before JSON.
No text after JSON.

Rules:
5 scenes.
4 seconds each.
20 seconds total.
9:16.
Natural Hinglish.
Maximum 10 spoken words per scene.
No speaker labels.
Same orange cat.
Scene 1 hook.
Scene 2 setup.
Scene 3 escalation.
Scene 4 peak.
Scene 5 surprise punchline.

SCHEMA:

{
"title":"string",
"hook":"string",
"language":"Hinglish",
"duration_sec":20,
"format":"9:16",
"character":{
"name":"Orange Cat",
"appearance":"string",
"personality":"string"
},
"scenes":[
{
"scene_number":1,
"duration_sec":4,
"visual_prompt":"string",
"dialogue":"string",
"action":"string"
},
{
"scene_number":2,
"duration_sec":4,
"visual_prompt":"string",
"dialogue":"string",
"action":"string"
},
{
"scene_number":3,
"duration_sec":4,
"visual_prompt":"string",
"dialogue":"string",
"action":"string"
},
{
"scene_number":4,
"duration_sec":4,
"visual_prompt":"string",
"dialogue":"string",
"action":"string"
},
{
"scene_number":5,
"duration_sec":4,
"visual_prompt":"string",
"dialogue":"string",
"action":"string"
}
]
}

SOURCE:
""" + json.dumps(original, ensure_ascii=False)

    raw2 = generate_repair(
        emergency_prompt,
        max_new_tokens=1200
    )

    print("\nRAW EMERGENCY OUTPUT:")
    print(raw2[:4000])

    plan = extract_json(raw2)


# ------------------------------------------------------------
# 10. FINAL SAFETY CHECK
# ------------------------------------------------------------

assert plan is not None, (
    "Qwen failed to produce JSON after two repair attempts. "
    "This is a model-output issue, not a GPU/Kaggle issue."
)

required = [
    "title",
    "hook",
    "language",
    "duration_sec",
    "format",
    "character",
    "scenes"
]

for key in required:
    assert key in plan, f"Missing field: {key}"

assert plan["language"] == "Hinglish"
assert plan["duration_sec"] == 20
assert plan["format"] == "9:16"

assert isinstance(plan["scenes"], list)
assert len(plan["scenes"]) == 5

# ------------------------------------------------------------
# 11. CONTROLLER LOCK
# ------------------------------------------------------------

plan["duration_sec"] = 20
plan["format"] = "9:16"
plan["language"] = "Hinglish"

for i, scene in enumerate(plan["scenes"], 1):

    scene["scene_number"] = i
    scene["duration_sec"] = 4

    # Remove accidental speaker prefixes
    dialogue = str(scene.get("dialogue", "")).strip()

    dialogue = re.sub(
        r"^[A-Za-z ]+:\s*",
        "",
        dialogue
    ).strip()

    scene["dialogue"] = dialogue

# ------------------------------------------------------------
# 12. FINAL QUALITY CHECK
# ------------------------------------------------------------

final_issues = []

for i, scene in enumerate(plan["scenes"], 1):

    words = scene["dialogue"].split()

    if len(words) > 10:
        final_issues.append(
            f"Scene {i}: {len(words)} dialogue words."
        )

if final_issues:

    print("\n⚠️ Final quality warnings:")
    for x in final_issues:
        print(" -", x)

# ------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------

with open(
    OUTPUT_PLAN,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        plan,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 14. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("🎉 KAGGLE-SAFE-11.5.1 PASS")
print("=" * 60)

print("Output:", OUTPUT_PLAN)
print("Title:", plan["title"])
print("Hook:", plan["hook"])
print("Scenes:", len(plan["scenes"]))
print("Duration:", plan["duration_sec"], "sec")
print("Format:", plan["format"])

print("\n🎬 FINAL DIALOGUES:")

for scene in plan["scenes"]:
    print(
        f'{scene["scene_number"]}. '
        f'{scene["dialogue"]}'
    )

print("\n✅ Creative plan locked.")
print("➡️ Next phase: actual moving AI video generation.")

🎯 PERSONAL AI — ROBUST REEL QUALITY JUDGE + AUTO-REPAIR


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Qwen loaded

🔍 QUALITY AUDIT
⚠️ Issues detected:
 - Total duration must be exactly 20 seconds.
 - Scene 1 duration must be exactly 4 seconds.
 - Scene 1 dialogue too long: 24 words.
 - Scene 2 duration must be exactly 4 seconds.
 - Scene 2 dialogue too long: 22 words.
 - Scene 3 duration must be exactly 4 seconds.
 - Scene 3 dialogue too long: 21 words.
 - Scene 4 duration must be exactly 4 seconds.
 - Scene 4 dialogue too long: 17 words.
 - Scene 5 duration must be exactly 4 seconds.
 - Scene 5 dialogue too long: 35 words.
 - Repeated phrase detected: 'bhai, seriously'
 - Repeated phrase detected: 'seriously'
 - Repeated phrase detected: 'you’re asking too much'
 - Scene 4 and Scene 5 are too similar.

🧠 Running first repair...

RAW REPAIR OUTPUT:
```json
{
  "title": "Cat's Complaint",
  "hook": "What's this cat saying?",
  "language": "Hinglish",
  "duration_seconds": 20,
  "format": "9:16",
  "character": {
    "name": "Orange Cat",
    "appearance": "orange cat with expressive f

AssertionError: Missing field: duration_sec

In [21]:
# ============================================================
# PATCH — CONTROLLER DEFAULTS
# ============================================================

# Missing top-level fields ko controller automatically fix karega

plan.setdefault("title", "Orange Cat Reel")
plan.setdefault("hook", plan.get("title", "Funny Cat"))
plan["language"] = "Hinglish"
plan["duration_sec"] = 20
plan["format"] = "9:16"

plan.setdefault(
    "character",
    {
        "name": "Orange Cat",
        "appearance": "realistic expressive orange cat",
        "personality": "funny, dramatic, mischievous"
    }
)

plan["character"].setdefault("name", "Orange Cat")
plan["character"].setdefault(
    "appearance",
    "realistic expressive orange cat"
)
plan["character"].setdefault(
    "personality",
    "funny, dramatic, mischievous"
)

# ------------------------------------------------------------
# FORCE 5 SCENES
# ------------------------------------------------------------

assert "scenes" in plan, "Missing scenes"

assert isinstance(plan["scenes"], list), "Scenes must be a list"

assert len(plan["scenes"]) >= 5, (
    f"Need at least 5 scenes, got {len(plan['scenes'])}"
)

plan["scenes"] = plan["scenes"][:5]

# ------------------------------------------------------------
# SCENE CONTROLLER
# ------------------------------------------------------------

for i, scene in enumerate(plan["scenes"], 1):

    scene["scene_number"] = i
    scene["duration_sec"] = 4

    scene.setdefault(
        "visual_prompt",
        "Realistic orange cat in an Indian middle-class home."
    )

    scene.setdefault(
        "dialogue",
        ""
    )

    scene.setdefault(
        "action",
        "The orange cat reacts naturally."
    )

    # Remove accidental speaker prefix
    scene["dialogue"] = re.sub(
        r"^[A-Za-z ]+:\s*",
        "",
        str(scene["dialogue"])
    ).strip()

# ------------------------------------------------------------
# FINAL HARD VALIDATION
# ------------------------------------------------------------

required = [
    "title",
    "hook",
    "language",
    "duration_sec",
    "format",
    "character",
    "scenes"
]

for key in required:
    assert key in plan, f"Missing field after controller repair: {key}"

assert plan["language"] == "Hinglish"
assert plan["duration_sec"] == 20
assert plan["format"] == "9:16"
assert len(plan["scenes"]) == 5

for i, scene in enumerate(plan["scenes"], 1):

    assert scene["scene_number"] == i
    assert scene["duration_sec"] == 4
    assert "visual_prompt" in scene
    assert "dialogue" in scene
    assert "action" in scene

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

with open(
    OUTPUT_PLAN,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        plan,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n" + "=" * 60)
print("🎉 CONTROLLER PATCH PASS")
print("=" * 60)

print("Title:", plan["title"])
print("Hook:", plan["hook"])
print("Language:", plan["language"])
print("Duration:", plan["duration_sec"])
print("Format:", plan["format"])
print("Scenes:", len(plan["scenes"]))

print("\n🎬 FINAL PLAN:")
for s in plan["scenes"]:
    print(
        f'{s["scene_number"]}. '
        f'[{s["duration_sec"]}s] '
        f'{s["dialogue"]}'
    )

print("\n💾 Saved:", OUTPUT_PLAN)


🎉 CONTROLLER PATCH PASS
Title: Cat's Complaint
Hook: What's this cat saying?
Language: Hinglish
Duration: 20
Format: 9:16
Scenes: 5

🎬 FINAL PLAN:
1. [4s] 'Yaar, bhai, seriously, why do I have to clean this mess? It’s already dirty! And you expect me to do it all day long?!'
2. [4s] 'Bhai, seriously, how can I clean when there’s no time for even a little bit of effort? This place is a disaster!'
3. [4s] 'Bhai, seriously, if I don’t clean now, I’ll just end up doing it all night! Look at this mop, it’s useless!'
4. [4s] 'Bhai, seriously, if I don’t get paid, I won’t clean anymore! You’re asking too much from me!'
5. [4s] 'Bhai, seriously, if I don’t get paid, I won’t clean anymore! You’re asking too much from me! But I’m going to make sure you know that I will clean my room properly from now on!'

💾 Saved: /kaggle/working/reel_director_quality_locked.json


In [22]:
# ============================================================
# KAGGLE-SAFE-11.5.2
# DETERMINISTIC CREATIVE QUALITY GATE
# ============================================================

import os
import json
import re

INPUT_PLAN = "/kaggle/working/reel_director_quality_locked.json"
OUTPUT_PLAN = "/kaggle/working/reel_director_production_ready.json"

print("🎯 PERSONAL AI — DETERMINISTIC CREATIVE QUALITY GATE")

assert os.path.exists(INPUT_PLAN), f"Missing: {INPUT_PLAN}"

with open(INPUT_PLAN, "r", encoding="utf-8") as f:
    plan = json.load(f)

# ------------------------------------------------------------
# CONTROLLER LOCK
# ------------------------------------------------------------

plan["language"] = "Hinglish"
plan["duration_sec"] = 20
plan["format"] = "9:16"

assert len(plan["scenes"]) == 5

for i, scene in enumerate(plan["scenes"], 1):
    scene["scene_number"] = i
    scene["duration_sec"] = 4

    scene["dialogue"] = re.sub(
        r"^[A-Za-z ]+:\s*",
        "",
        str(scene.get("dialogue", ""))
    ).strip()

# ------------------------------------------------------------
# QUALITY AUDIT
# ------------------------------------------------------------

issues = []

# 1. Dialogue length
for i, scene in enumerate(plan["scenes"], 1):

    words = re.findall(
        r"\b[\w’']+\b",
        scene["dialogue"]
    )

    if len(words) > 10:
        issues.append(
            f"Scene {i}: dialogue has {len(words)} words (>10)"
        )

# 2. Repeated phrases
dialogues = [
    s["dialogue"].lower()
    for s in plan["scenes"]
]

full_text = " ".join(dialogues)

for phrase in [
    "bhai, seriously",
    "seriously",
    "i can't believe",
    "this is so frustrating",
    "you're asking too much",
    "you’re asking too much"
]:
    if full_text.count(phrase) > 1:
        issues.append(
            f"Repeated phrase: {phrase}"
        )

# 3. Duplicate/similar scenes
for i in range(len(dialogues)):
    for j in range(i + 1, len(dialogues)):

        a = set(re.findall(r"\w+", dialogues[i]))
        b = set(re.findall(r"\w+", dialogues[j]))

        if a and b:
            similarity = len(a & b) / len(a | b)

            if similarity >= 0.55:
                issues.append(
                    f"Scenes {i+1} and {j+1} too similar "
                    f"({similarity:.2f})"
                )

# 4. Speaker prefixes
for i, scene in enumerate(plan["scenes"], 1):

    if re.match(
        r"^(orange cat|cat|character)\s*:",
        scene["dialogue"],
        re.I
    ):
        issues.append(
            f"Scene {i}: speaker prefix detected"
        )

# 5. Empty dialogue
for i, scene in enumerate(plan["scenes"], 1):

    if not scene["dialogue"].strip():
        issues.append(
            f"Scene {i}: empty dialogue"
        )

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("\n🔍 QUALITY REPORT")
print("-" * 60)

if issues:

    print("❌ PLAN REJECTED")
    print()

    for issue in issues:
        print(" •", issue)

    print()
    print("Reason:")
    print("The current AI-generated creative plan is NOT")
    print("safe to send to the video-generation backend.")

    print("\n➡️ Next action:")
    print("Regenerate the creative plan using the stricter")
    print("production prompt.")

else:

    # --------------------------------------------------------
    # SAVE PRODUCTION PLAN
    # --------------------------------------------------------

    with open(
        OUTPUT_PLAN,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            plan,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("✅ PLAN PASSED ALL CREATIVE GATES")
    print()
    print("🎬 Production Plan:")

    for scene in plan["scenes"]:
        print(
            f'{scene["scene_number"]}. '
            f'{scene["dialogue"]}'
        )

    print()
    print("💾 Saved:")
    print(OUTPUT_PLAN)

    print()
    print("🎉 KAGGLE-SAFE-11.5.2 PASS")
    print("➡️ Ready for video-generation backend.")

🎯 PERSONAL AI — DETERMINISTIC CREATIVE QUALITY GATE

🔍 QUALITY REPORT
------------------------------------------------------------
❌ PLAN REJECTED

 • Scene 1: dialogue has 24 words (>10)
 • Scene 2: dialogue has 22 words (>10)
 • Scene 3: dialogue has 21 words (>10)
 • Scene 4: dialogue has 17 words (>10)
 • Scene 5: dialogue has 35 words (>10)
 • Repeated phrase: bhai, seriously
 • Repeated phrase: seriously
 • Repeated phrase: you’re asking too much
 • Scenes 4 and 5 too similar (0.56)

Reason:
The current AI-generated creative plan is NOT
safe to send to the video-generation backend.

➡️ Next action:
Regenerate the creative plan using the stricter
production prompt.


In [23]:
# ============================================================
# KAGGLE-SAFE-11.5.3
# AUTO-REGENERATE REJECTED REEL
# ============================================================

import os
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "/kaggle/input/datasets/vikaschandoliya/personal-ai-qwen"
INPUT_PLAN = "/kaggle/working/reel_director_quality_locked.json"
OUTPUT_PLAN = "/kaggle/working/reel_director_production_ready.json"

print("🎯 PERSONAL AI — AUTO-REGENERATE PRODUCTION CREATIVE")

# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

print("✅ Qwen loaded")

# ------------------------------------------------------------
# LOAD REJECTED PLAN
# ------------------------------------------------------------

with open(INPUT_PLAN, "r", encoding="utf-8") as f:
    old_plan = json.load(f)

# ------------------------------------------------------------
# STRICT CREATIVE PROMPT
# ------------------------------------------------------------

prompt = """
Create a NEW production-ready Instagram Reel.

IMPORTANT:
You are generating a SHORT comedy reel, NOT an essay.

RETURN ONLY VALID JSON.
NO markdown.
NO explanation.
NO text before JSON.
NO text after JSON.
NO speaker labels.

CREATIVE CONCEPT:

A realistic expressive orange cat lives in an Indian middle-class home.

The cat is frustrated because everyone expects it to clean the messy room.

The comedy should escalate and end with a surprising punchline.

LANGUAGE:
Natural Indian Hinglish.

DIALOGUE RULES — EXTREMELY IMPORTANT:

Each scene dialogue MUST contain 6 to 10 words maximum.

Target approximately 7-8 words.

Do NOT exceed 10 words.

Each scene should be ONE short spoken sentence.

Never use:
"bhai, seriously"
"seriously"
"I can't believe"
"This is so frustrating"
"you're asking too much"

Do not repeat the same joke.

Do not use speaker names inside dialogue.

Do not write long sentences.

STORY STRUCTURE:

Scene 1:
Strong funny hook.

Scene 2:
Cat discovers the cleaning problem.

Scene 3:
Cat gets an unexpected funny idea.

Scene 4:
Cat performs a simple funny action.

Scene 5:
Unexpected punchline that changes the situation.

VISUAL RULES:

Realistic orange cat.
Indian middle-class home.
Natural lighting.
Cinematic realistic appearance.
Simple physically possible actions.
No complicated transformations.
No extra characters required.

CONTINUITY:

Same orange cat in every scene.
Same home.
Same clothing/appearance.
Same personality.

TIMING:

Exactly 5 scenes.
Exactly 4 seconds each.
Exactly 20 seconds total.
9:16 vertical.

SCENE ACTIONS:

Use simple actions such as:
looking at camera,
picking up cloth,
moving a small object,
dropping mop,
sitting on chair,
walking away,
staring at camera.

Avoid:
flying,
teleportation,
body transformation,
multiple characters,
complex choreography.

EXACT JSON SCHEMA:

{
  "title": "string",
  "hook": "string",
  "language": "Hinglish",
  "duration_sec": 20,
  "format": "9:16",
  "character": {
    "name": "Orange Cat",
    "appearance": "realistic expressive orange cat",
    "personality": "funny, dramatic, mischievous"
  },
  "scenes": [
    {
      "scene_number": 1,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "6-10 words only",
      "action": "simple physical action"
    },
    {
      "scene_number": 2,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "6-10 words only",
      "action": "simple physical action"
    },
    {
      "scene_number": 3,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "6-10 words only",
      "action": "simple physical action"
    },
    {
      "scene_number": 4,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "6-10 words only",
      "action": "simple physical action"
    },
    {
      "scene_number": 5,
      "duration_sec": 4,
      "visual_prompt": "string",
      "dialogue": "6-10 words only",
      "action": "simple physical action"
    }
  ]
}

REJECTED PLAN:
""" + json.dumps(old_plan, ensure_ascii=False)

# ------------------------------------------------------------
# GENERATE
# ------------------------------------------------------------

messages = [
    {
        "role": "system",
        "content": (
            "You are a professional short-form comedy "
            "creative director. Return JSON only."
        )
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

print("🧠 Generating new creative plan...")

with torch.no_grad():

    output = model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=False,
        temperature=None,
        top_p=None
    )

generated = output[0][inputs["input_ids"].shape[1]:]

raw = tokenizer.decode(
    generated,
    skip_special_tokens=True
).strip()

print("\nRAW OUTPUT:")
print(raw[:5000])

# ------------------------------------------------------------
# EXTRACT JSON
# ------------------------------------------------------------

def extract_json(text):

    text = re.sub(
        r"```(?:json)?",
        "",
        text,
        flags=re.I
    )

    text = text.replace("```", "").strip()

    start = text.find("{")
    end = text.rfind("}")

    if start < 0 or end <= start:
        return None

    candidate = text[start:end + 1]

    # trailing commas
    candidate = re.sub(
        r",\s*([}\]])",
        r"\1",
        candidate
    )

    # smart quotes
    candidate = (
        candidate
        .replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )

    try:
        return json.loads(candidate)
    except Exception:
        return None


plan = extract_json(raw)

assert plan is not None, (
    "Qwen did not return valid JSON."
)

# ------------------------------------------------------------
# CONTROLLER LOCK
# ------------------------------------------------------------

plan["language"] = "Hinglish"
plan["duration_sec"] = 20
plan["format"] = "9:16"

assert "scenes" in plan
assert isinstance(plan["scenes"], list)
assert len(plan["scenes"]) >= 5

plan["scenes"] = plan["scenes"][:5]

for i, scene in enumerate(plan["scenes"], 1):

    scene["scene_number"] = i
    scene["duration_sec"] = 4

    scene["dialogue"] = re.sub(
        r"^(orange cat|cat|character)\s*:\s*",
        "",
        str(scene.get("dialogue", "")),
        flags=re.I
    ).strip()

    scene.setdefault(
        "visual_prompt",
        "Realistic expressive orange cat in an Indian middle-class home."
    )

    scene.setdefault(
        "action",
        "The orange cat reacts naturally."
    )

# ------------------------------------------------------------
# FINAL QUALITY GATE
# ------------------------------------------------------------

issues = []

dialogues = []

for i, scene in enumerate(plan["scenes"], 1):

    dialogue = scene["dialogue"]

    words = re.findall(
        r"\b[\w’']+\b",
        dialogue
    )

    dialogues.append(dialogue.lower())

    if len(words) < 6:
        issues.append(
            f"Scene {i}: only {len(words)} words."
        )

    if len(words) > 10:
        issues.append(
            f"Scene {i}: {len(words)} words (>10)."
        )

# repeated forbidden phrases
full_text = " ".join(dialogues)

for phrase in [
    "bhai, seriously",
    "seriously",
    "i can't believe",
    "this is so frustrating",
    "you're asking too much",
    "you’re asking too much"
]:

    if phrase in full_text:
        issues.append(
            f"Forbidden/repeated phrase: {phrase}"
        )

# scene similarity
for i in range(5):
    for j in range(i + 1, 5):

        a = set(re.findall(r"\w+", dialogues[i]))
        b = set(re.findall(r"\w+", dialogues[j]))

        if a and b:

            similarity = len(a & b) / len(a | b)

            if similarity >= 0.55:
                issues.append(
                    f"Scenes {i+1} and {j+1} too similar "
                    f"({similarity:.2f})."
                )

# ------------------------------------------------------------
# SAVE ONLY IF PASS
# ------------------------------------------------------------

if issues:

    print("\n❌ PRODUCTION CREATIVE REJECTED")

    for issue in issues:
        print(" •", issue)

    print("\nDo NOT send this plan to video generation.")

    raise AssertionError(
        "Creative quality gate failed."
    )

with open(
    OUTPUT_PLAN,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        plan,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# PASS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("🎉 KAGGLE-SAFE-11.5.3 PASS")
print("=" * 60)

print("Title:", plan["title"])
print("Hook:", plan["hook"])
print("Duration:", plan["duration_sec"])
print("Format:", plan["format"])

print("\n🎬 PRODUCTION DIALOGUES:")

for scene in plan["scenes"]:
    print(
        f'{scene["scene_number"]}. '
        f'{scene["dialogue"]}'
    )

print("\n💾 Saved:")
print(OUTPUT_PLAN)

print("\n🚀 CREATIVE PLAN IS READY FOR VIDEO BACKEND.")

🎯 PERSONAL AI — AUTO-REGENERATE PRODUCTION CREATIVE


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Qwen loaded
🧠 Generating new creative plan...

RAW OUTPUT:
```json
{
  "title": "Orange Cat's Frustration",
  "hook": "Why does the orange cat keep getting frustrated?",
  "language": "Hinglish",
  "duration_sec": 20,
  "format": "9:16",
  "character": {
    "name": "Orange Cat",
    "appearance": "realistic expressive orange cat",
    "personality": "funny, dramatic, mischievous"
  },
  "scenes": [
    {
      "scene_number": 1,
      "duration_sec": 4,
      "visual_prompt": "Orange cat standing on a cluttered floor, looking annoyed.",
      "dialogue": "‘Yaar, bhai, seriously, why do I have to clean this mess? It’s already dirty! And you expect me to do it all day long?!’",
      "action": "Orange Cat throws a small toy across the room."
    },
    {
      "scene_number": 2,
      "duration_sec": 4,
      "visual_prompt": "Orange cat sitting on a chair, fur matted, looking frustrated.",
      "dialogue": "‘Bhai, seriously, how can I clean when there’s no time for even a little bit

AssertionError: Creative quality gate failed.